# 06 외부 데이터 수집 및 전처리

원본 Colab의 데이터 격자화 단계 중 외부 데이터 수집과 전처리 셀만 분리한 노트북입니다. 경로는 원본 프로젝트 기준으로 유지했으므로, 다른 실행자는 `ROOT_PATH`만 본인 Google Drive 경로에 맞게 수정하면 됩니다.

In [ ]:
# Colab 단독 실행용 기본 경로 설정
from google.colab import drive
drive.mount('/content/drive')

import os

ROOT_PATH = "/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/"
DATA_PATH = ROOT_PATH + "data/"
PROCESSED_PATH = ROOT_PATH + "preprocessed_data/"

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(os.path.join(PROCESSED_PATH, "additional_data"), exist_ok=True)

print("ROOT_PATH:", ROOT_PATH)
print("DATA_PATH:", DATA_PATH)
print("PROCESSED_PATH:", PROCESSED_PATH)


## 1. 공장/저유소/대리점 위치 데이터

대한석유협회 회원사 페이지에서 정유사별 공장, 저유소, 대리점 주소를 수집하고, Google Maps UI 자동화로 좌표를 보강합니다.

### 1-1. 회원사 시설 데이터 수집

In [ ]:
import re
import time
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

# =========================
# 환경설정
# =========================
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

COMPANY_MAP = {
    1: {"상표": "sk",    "overview": "https://www.petroleum.or.kr/association/member_1",
        "storage": "https://www.petroleum.or.kr/association/member_1_4",
        "agency":  "https://www.petroleum.or.kr/association/member_1_6"},
    2: {"상표": "gs",    "overview": "https://www.petroleum.or.kr/association/member_2",
        "storage": "https://www.petroleum.or.kr/association/member_2_4",
        "agency":  "https://www.petroleum.or.kr/association/member_2_6"},
    3: {"상표": "s-oil", "overview": "https://www.petroleum.or.kr/association/member_3",
        "storage": "https://www.petroleum.or.kr/association/member_3_4",
        "agency":  "https://www.petroleum.or.kr/association/member_3_6"},
    4: {"상표": "hd",    "overview": "https://www.petroleum.or.kr/association/member_4",
        "storage": "https://www.petroleum.or.kr/association/member_4_4",
        "agency":  "https://www.petroleum.or.kr/association/member_4_6"},
}

session = requests.Session()
session.headers.update(HEADERS)

# =========================
# 함수 정의
# =========================
def log(msg: str) -> None:
    print(msg)

def clean_text(x) -> str:
    if x is None:
        return ""
    x = str(x).replace("\xa0", " ").replace("：", ":")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_address(x: str) -> str:
    x = clean_text(x)
    x = re.split(r"\bTEL\b|\bFAX\b|전화번호|전화|팩스", x, maxsplit=1, flags=re.IGNORECASE)[0]
    x = x.strip(" /,-")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def add_or_replace_query(url: str, **params) -> str:
    parsed = urlparse(url)
    query = parse_qs(parsed.query)
    for k, v in params.items():
        query[k] = [str(v)]
    new_query = urlencode(query, doseq=True)
    return urlunparse(parsed._replace(query=new_query))

def candidate_urls(base_url: str, page: int | None = None):
    candidates = []
    for url in [base_url, base_url.replace("/association/", "/kor/association/")]:
        if page is not None:
            url = add_or_replace_query(url, page=page)
        if url not in candidates:
            candidates.append(url)
    return candidates

def fetch_html(url: str, sleep_sec: float = 0.2) -> str:
    resp = session.get(url, timeout=30)
    resp.raise_for_status()
    if not resp.encoding:
        resp.encoding = resp.apparent_encoding or "utf-8"
    time.sleep(sleep_sec)
    return resp.text

def fetch_first_success(urls, retries_per_url: int = 2) -> str:
    last_error = None
    for url in urls:
        for attempt in range(1, retries_per_url + 1):
            try:
                log(f"    요청 시도: {url} (attempt={attempt})")
                return fetch_html(url)
            except Exception as e:
                last_error = e
                log(f"    요청 실패: {url} | {type(e).__name__}: {e}")
                time.sleep(0.5)
    if last_error:
        raise last_error
    raise RuntimeError("요청 가능한 URL이 없습니다.")

def get_soup_from_candidates(urls) -> BeautifulSoup:
    html = fetch_first_success(urls)
    return BeautifulSoup(html, "html.parser")

def extract_page_count(soup: BeautifulSoup) -> int:
    pages = []
    for a in soup.select(".paging a[href]"):
        txt = clean_text(a.get_text(" ", strip=True))
        href = a.get("href", "")
        if txt.isdigit():
            pages.append(int(txt))
        m = re.search(r"[?&]page=(\d+)", href)
        if m:
            pages.append(int(m.group(1)))
    return max(pages) if pages else 1

def split_factory_cell(td_text: str, default_name: str = "공장"):
    lines = [clean_text(line) for line in td_text.split("\n")]
    lines = [normalize_address(line) for line in lines if normalize_address(line)]

    results = []
    for line in lines:
        name = default_name
        addr = line

        if " - " in line:
            left, right = line.split(" - ", 1)
            if normalize_address(right):
                name = clean_text(left) or default_name
                addr = normalize_address(right)
        elif ":" in line:
            left, right = line.split(":", 1)
            if normalize_address(right):
                name = clean_text(left) or default_name
                addr = normalize_address(right)

        results.append((name, addr))

    return results

def parse_company_overview(brand: str, url: str):
    log(f"[{brand}][회사개요] 요청 시작")
    soup = get_soup_from_candidates(candidate_urls(url))
    rows = []

    table = soup.select_one("div.mem_com div.txt table.tbl-list") or soup.select_one("table.tbl-list")
    if table:
        for tr in table.select("tr"):
            th_tag = tr.select_one("th")
            td_tag = tr.select_one("td")
            if not th_tag or not td_tag:
                continue

            th = clean_text(th_tag.get_text(" ", strip=True))
            td_text = td_tag.get_text("\n", strip=True)

            if "공장" in th:
                default_name = th
                for name, addr in split_factory_cell(td_text, default_name=default_name):
                    if addr:
                        rows.append({
                            "상표": brand,
                            "구분": "공장",
                            "이름": name,
                            "주소": addr
                        })

    if not rows:
        log(f"[{brand}][회사개요] 테이블 기반 파싱 실패 -> 텍스트 fallback 시도")
        text_lines = [
            clean_text(x)
            for x in soup.get_text("\n", strip=True).split("\n")
            if clean_text(x)
        ]
        for line in text_lines:
            line_upper = line.upper()
            if ("공장" in line) and ("TEL" in line_upper or "FAX" in line_upper):
                default_name = "공장"
                if line.startswith("본사(공장)"):
                    default_name = "본사(공장)"
                    line = line.replace("본사(공장)", "", 1).strip()
                elif line.startswith("공장"):
                    line = line.replace("공장", "", 1).strip()

                addr = normalize_address(line)
                if addr:
                    rows.append({
                        "상표": brand,
                        "구분": "공장",
                        "이름": default_name,
                        "주소": addr
                    })

    log(f"[{brand}][회사개요] 추출 행 수: {len(rows)}")
    return rows

def parse_board_table(html: str, brand: str, category: str):
    soup = BeautifulSoup(html, "html.parser")
    table = soup.select_one("#sb-list table.tbl-list") or soup.select_one("table.tbl-list")
    if not table:
        return []

    headers = [clean_text(th.get_text(" ", strip=True)) for th in table.select("thead th")]
    header_index = {h: i for i, h in enumerate(headers)}

    rows = []
    for tr in table.select("tbody tr"):
        tds = tr.select("td")
        values = [clean_text(td.get_text(" ", strip=True)) for td in tds]

        if not values or all(v == "" for v in values):
            continue

        if category == "저유소":
            name = values[header_index["저유소명"]] if "저유소명" in header_index else values[0]
            addr = values[header_index["소재지"]] if "소재지" in header_index else values[1]
        else:
            name = values[header_index["대리점명"]] if "대리점명" in header_index else values[0]
            addr = values[header_index["소재지"]] if "소재지" in header_index else values[2]

        name = clean_text(name)
        addr = normalize_address(addr)

        if name and addr:
            rows.append({
                "상표": brand,
                "구분": category,
                "이름": name,
                "주소": addr
            })

    return rows

def collect_board(brand: str, category: str, url: str):
    log(f"[{brand}][{category}] 1페이지 요청 시작")
    first_html = fetch_first_success(candidate_urls(url, page=1))
    first_soup = BeautifulSoup(first_html, "html.parser")
    total_pages = extract_page_count(first_soup)
    log(f"[{brand}][{category}] 총 페이지 수 추정: {total_pages}")

    rows = []
    seen = set()

    for page in range(1, total_pages + 1):
        if page == 1:
            html = first_html
        else:
            html = fetch_first_success(candidate_urls(url, page=page))

        page_rows = parse_board_table(html, brand=brand, category=category)
        new_count = 0

        for row in page_rows:
            key = (row["상표"], row["구분"], row["이름"], row["주소"])
            if key not in seen:
                seen.add(key)
                rows.append(row)
                new_count += 1

        log(f"[{brand}][{category}] page={page} | 수집={len(page_rows)} | 신규반영={new_count}")

        if page > 1 and len(page_rows) == 0:
            log(f"[{brand}][{category}] page={page} 에서 빈 결과 -> 이후 페이지 중단")
            break

    log(f"[{brand}][{category}] 최종 누적 행 수: {len(rows)}")
    return rows

# =========================
# 사용자 설정 및 실행
# =========================
all_rows = []

for company_no, info in COMPANY_MAP.items():
    brand = info["상표"]
    log("=" * 120)
    log(f"[시작] company_no={company_no}, 상표={brand}")

    # 1) 회사개요 -> 공장
    all_rows.extend(parse_company_overview(brand, info["overview"]))

    # 2) 저유소
    all_rows.extend(collect_board(brand, "저유소", info["storage"]))

    # 3) 대리점
    all_rows.extend(collect_board(brand, "대리점", info["agency"]))

final_df = pd.DataFrame(all_rows, columns=["상표", "구분", "이름", "주소"]).drop_duplicates().reset_index(drop=True)

# 정렬
brand_order = {"sk": 0, "gs": 1, "s-oil": 2, "hd": 3}
type_order = {"공장": 0, "저유소": 1, "대리점": 2}

final_df["_brand_order"] = final_df["상표"].map(brand_order)
final_df["_type_order"] = final_df["구분"].map(type_order)
final_df = (
    final_df.sort_values(["_brand_order", "_type_order", "이름", "주소"])
            .drop(columns=["_brand_order", "_type_order"])
            .reset_index(drop=True)
)

log("=" * 120)
log(f"[완료] 최종 데이터프레임 행 수: {len(final_df)}")
log(f"[완료] 상표별 건수:\n{final_df['상표'].value_counts(dropna=False)}")
log(f"[완료] 구분별 건수:\n{final_df['구분'].value_counts(dropna=False)}")

display(final_df)

final_df.to_csv(DATA_PATH + 'facility_data.csv', index=False, encoding='utf-8-sig')

### 1-2. 경도/위도 크롤링

In [ ]:
# =========================================================
# Google Maps UI 자동화 기반 경도/위도 수집 - Enhanced 1 Cell
# 구성:
#   1) 환경설정
#   2) 함수정의
#   3) 사용자설정 및 실행
#
# 입력:
#   DATA_PATH + 'facility_data.csv'
#   필수컬럼: 상표, 구분, 이름, 주소
#
# 출력:
#   PROCESSED_PATH/additional_data/1 facility_location_data.csv
#
# 최종 변수:
#   result_df, not_nan_df, nan_df
# =========================================================

# =========================================================
# 1) 환경설정
# =========================================================
import os
import re
import sys
import time
import random
import asyncio
import subprocess
import importlib.util
import unicodedata
from urllib.parse import quote
from difflib import SequenceMatcher

import pandas as pd
import numpy as np

print("[0/3] 환경 준비 시작")

def run_cmd(cmd):
    print(f"[CMD] {' '.join(cmd)}")
    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if result.stdout:
        print(result.stdout[-2500:])
    if result.returncode != 0:
        raise RuntimeError(f"명령 실패: {' '.join(cmd)}")

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        print(f"[PKG] {pip_name} 설치")
        run_cmd([sys.executable, "-m", "pip", "install", "-q", pip_name])
    else:
        print(f"[PKG] {pip_name} 이미 설치됨")

print("[1/3] 시스템/패키지 설치")
run_cmd(["apt-get", "update", "-y"])
run_cmd([
    "apt-get", "install", "-y",
    "libatk1.0-0",
    "libatk-bridge2.0-0",
    "libgtk-3-0",
    "libnss3",
    "libxss1",
    "libasound2",
    "libxshmfence1",
    "libgbm1",
    "libxcomposite1",
    "libxdamage1",
    "libxrandr2",
    "libxkbcommon0",
    "libpango-1.0-0",
    "libpangocairo-1.0-0",
    "libcairo2",
    "libatspi2.0-0",
    "libcups2",
    "libdbus-1-3",
    "libdrm2",
    "libxfixes3",
    "libx11-xcb1",
    "libxcb1",
    "libxext6",
    "libx11-6",
    "libglib2.0-0",
    "fonts-liberation",
    "ca-certificates",
    "xdg-utils",
    "wget",
])

ensure_package("playwright")
ensure_package("nest_asyncio")
ensure_package("tqdm")

print("[2/3] Playwright Chromium 설치")
run_cmd([sys.executable, "-m", "playwright", "install", "chromium"])

import nest_asyncio
nest_asyncio.apply()

from tqdm.auto import tqdm
from playwright.async_api import async_playwright


# =========================================================
# 2) 함수정의
# =========================================================
FULLWIDTH_NUM_MAP = str.maketrans("０１２３４５６７８９", "0123456789")

NUM_WORD_TO_DIGIT = {
    "일": "1",
    "이": "2",
    "삼": "3",
    "사": "4",
    "오": "5",
    "육": "6",
    "칠": "7",
    "팔": "8",
    "구": "9",
}
DIGIT_TO_NUM_WORD = {v: k for k, v in NUM_WORD_TO_DIGIT.items()}

PREFIX_REPLACEMENTS = [
    (r"^서울시\b", "서울특별시"),
    (r"^서울\b", "서울특별시"),
    (r"^부산시\b", "부산광역시"),
    (r"^부산\b", "부산광역시"),
    (r"^대구시\b", "대구광역시"),
    (r"^대구\b", "대구광역시"),
    (r"^인천시\b", "인천광역시"),
    (r"^인천\b", "인천광역시"),
    (r"^광주시\b", "광주광역시"),
    (r"^광주\b", "광주광역시"),
    (r"^대전시\b", "대전광역시"),
    (r"^대전\b", "대전광역시"),
    (r"^울산시\b", "울산광역시"),
    (r"^울산\b", "울산광역시"),
    (r"^세종시\b", "세종특별자치시"),
    (r"^경기\b", "경기도"),
    (r"^강원\b", "강원도"),
    (r"^충북\b", "충청북도"),
    (r"^충남\b", "충청남도"),
    (r"^전북\b", "전라북도"),
    (r"^전남\b", "전라남도"),
    (r"^경북\b", "경상북도"),
    (r"^경남\b", "경상남도"),
    (r"^제주도\b", "제주특별자치도"),
    (r"^제주\b", "제주특별자치도"),
]

LONG_TO_SHORT_REGION = {
    "서울특별시": "서울",
    "부산광역시": "부산",
    "대구광역시": "대구",
    "인천광역시": "인천",
    "광주광역시": "광주",
    "대전광역시": "대전",
    "울산광역시": "울산",
    "세종특별자치시": "세종",
    "경기도": "경기",
    "강원도": "강원",
    "충청북도": "충북",
    "충청남도": "충남",
    "전라북도": "전북",
    "전라남도": "전남",
    "경상북도": "경북",
    "경상남도": "경남",
    "제주특별자치도": "제주",
}

BRAND_ALIASES = {
    "SK": ["SK", "SK에너지", "SK가스", "에스케이"],
    "GS": ["GS", "GS칼텍스", "GS에너지", "지에스"],
    "S-OIL": ["S-OIL", "S OIL", "SOIL", "에쓰오일", "에스오일"],
    "HD": ["HD", "현대오일뱅크", "현대오일", "현대"],
}

CORP_TOKENS = ["(주)", "㈜", "주식회사", "유한회사", "합자회사", "합명회사"]

UI_NOISE_TOKENS = [
    "저장됨", "최근 항목", "앱 설치", "경로", "저장", "주변",
    "휴대전화로 보내기", "공유", "로그인", "Google 지도 최대한 활용하기",
]

def dedupe_keep_order(items):
    out = []
    seen = set()
    for item in items:
        if item in ("", None):
            continue
        if item not in seen:
            seen.add(item)
            out.append(item)
    return out

def clean_text(x):
    x = "" if pd.isna(x) else str(x)
    x = unicodedata.normalize("NFKC", x)
    x = x.translate(FULLWIDTH_NUM_MAP)
    x = x.replace("\u3000", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_brand(v):
    s = clean_text(v).lower().replace("_", "-")
    if s in ("sk", "sk에너지", "에스케이"):
        return "SK"
    if s in ("gs", "gs칼텍스", "지에스"):
        return "GS"
    if s in ("s-oil", "soil", "s oil", "에쓰오일", "에스오일"):
        return "S-OIL"
    if s in ("hd", "현대오일뱅크", "현대오일"):
        return "HD"
    return clean_text(v).upper()

def normalize_target(v):
    return clean_text(v)

def strip_corp_tokens(name: str) -> str:
    s = clean_text(name)
    for tok in CORP_TOKENS:
        s = s.replace(tok, " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_name_for_match(name: str) -> str:
    s = strip_corp_tokens(name)
    s = re.sub(r"\([^)]*\)", " ", s)
    s = re.sub(r"[^0-9A-Za-z가-힣]+", "", s)
    return s.upper()

def tokenize_name(name: str):
    s = strip_corp_tokens(name)
    s = re.sub(r"[()\/,_\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    tokens = []
    for tok in s.split():
        tok2 = re.sub(r"[^0-9A-Za-z가-힣]+", "", tok)
        if len(tok2) >= 2:
            tokens.append(tok2.upper())
    return tokens

def name_match_score(candidate_name: str, target_name: str):
    cand_norm = normalize_name_for_match(candidate_name)
    target_norm = normalize_name_for_match(target_name)

    if not cand_norm or not target_norm:
        return 0, "empty_name"

    if cand_norm == target_norm:
        return 100, "name_exact"
    if cand_norm in target_norm or target_norm in cand_norm:
        return 80, "name_contains"

    ratio = SequenceMatcher(None, cand_norm, target_norm).ratio()
    cand_tokens = set(tokenize_name(candidate_name))
    target_tokens = set(tokenize_name(target_name))
    overlap = len(cand_tokens & target_tokens)

    if overlap >= 3 and ratio >= 0.75:
        return 70, "name_overlap_3"
    if overlap >= 2 and ratio >= 0.72:
        return 58, "name_overlap_2"
    if overlap >= 1 and ratio >= 0.80:
        return 45, "name_overlap_1"
    if ratio >= 0.90:
        return 55, "name_ratio_090"
    if ratio >= 0.82:
        return 40, "name_ratio_082"
    return 0, "name_no_match"

def brand_match_score(candidate_name: str, brand: str):
    cand = normalize_name_for_match(candidate_name)
    for alias in BRAND_ALIASES.get(normalize_brand(brand), []):
        alias_norm = normalize_name_for_match(alias)
        if alias_norm and alias_norm in cand:
            return 10, "brand_match"
    return 0, "brand_no_match"

def expand_region_prefix(addr: str) -> str:
    a = clean_text(addr)
    for pattern, replacement in PREFIX_REPLACEMENTS:
        a = re.sub(pattern, replacement, a)
    return clean_text(a)

def shorten_region_prefix(addr: str) -> str:
    a = clean_text(addr)
    for long_name, short_name in LONG_TO_SHORT_REGION.items():
        a = re.sub(rf"^{re.escape(long_name)}\b", short_name, a)
    return clean_text(a)

def normalize_admin_unit_numbers_to_digits(text: str) -> str:
    s = clean_text(text)
    if not s:
        return ""

    pattern = re.compile(r"([가-힣A-Za-z]+?)\s*([123456789일이삼사오육칠팔구])\s*(동|가|리|통|반)\b")

    def repl(m):
        prefix, num, suffix = m.group(1), m.group(2), m.group(3)
        digit = NUM_WORD_TO_DIGIT.get(num, num)
        return f"{prefix}{digit}{suffix}"

    prev = None
    while prev != s:
        prev = s
        s = pattern.sub(repl, s)

    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_query_text_loose(text: str) -> str:
    a = clean_text(text)
    if not a:
        return ""

    a = re.sub(r"(\d+)\s*의\s*(\d+)", r"\1-\2", a)
    a = expand_region_prefix(a)
    a = normalize_admin_unit_numbers_to_digits(a)

    a = a.replace("창학1동", "청학1동")
    a = a.replace("소홀읍", "소흘읍")
    a = a.replace("마산시 합포구", "창원시 마산합포구")
    a = a.replace("마산시 회원구", "창원시 마산회원구")
    a = a.replace("진해시", "창원시 진해구")
    a = a.replace("김해시삼계동", "김해시 삼계동")
    a = a.replace("서산시부석면", "서산시 부석면")
    a = a.replace("아산시염치읍", "아산시 염치읍")
    a = a.replace("천안시청당동", "천안시 청당동")
    a = a.replace("제주시 용담 2동", "제주시 용담2동")
    a = a.replace("일도 2동", "일도2동")
    a = a.replace("가좌 4동", "가좌4동")
    a = a.replace("칠성동 2가", "칠성동2가")
    a = a.replace("봉래동 1가", "봉래동1가")
    a = a.replace("대구시 북구 칠성동 2가 67의1", "대구광역시 북구 칠성동2가 67-1")

    a = re.sub(r"\s+", " ", a).strip()
    return a

def normalize_address(addr: str) -> str:
    return normalize_query_text_loose(addr)

def simplify_address(addr: str) -> str:
    a = normalize_address(addr)
    if not a:
        return ""

    a = re.sub(r"\([^)]*\)", " ", a)
    a = re.sub(r"\b\d+\s*층\b", " ", a, flags=re.I)
    a = re.sub(r"\b\d+(?:-\d+)?\s*호\b", " ", a, flags=re.I)
    a = re.sub(r"\b\d+(?:-\d+)?\s*실\b", " ", a, flags=re.I)
    a = re.sub(r"\b\d+\s*F\b", " ", a, flags=re.I)
    a = re.sub(r"\bB\/D\b", " ", a, flags=re.I)
    a = re.sub(r"\s*외\s*\d+\s*필지.*$", " ", a)
    a = re.sub(r"\s*외\d+\s*필지.*$", " ", a)
    a = re.sub(r"\s*외\s*\d+.*$", " ", a)
    a = re.sub(r"\s*\d+-\d+\s*블록.*$", " ", a)
    a = a.replace("번지", " ")
    a = re.sub(r"\s+", " ", a).strip()
    return a

def generate_query_candidates_original(addr: str):
    norm = normalize_address(addr)
    simp = simplify_address(addr)
    out = [norm, simp]
    if norm:
        out.append(f"대한민국 {norm}")
    if simp:
        out.append(f"대한민국 {simp}")
    return dedupe_keep_order([clean_text(x) for x in out if clean_text(x)])

def generate_admin_number_variants(text: str):
    s = clean_text(text)
    if not s:
        return []

    out = [s]
    pattern = re.compile(r"([가-힣A-Za-z]+?)\s*([123456789일이삼사오육칠팔구])\s*(동|가|리|통|반)\b")

    def add_variants(src):
        items = [src]
        for m in list(pattern.finditer(src)):
            prefix, num, suffix = m.group(1), m.group(2), m.group(3)
            digit = NUM_WORD_TO_DIGIT.get(num, num)
            word = DIGIT_TO_NUM_WORD.get(digit, num)
            old = m.group(0)

            items.append(src.replace(old, f"{prefix}{digit}{suffix}", 1))
            items.append(src.replace(old, f"{prefix}{word}{suffix}", 1))
            items.append(src.replace(old, f"{prefix} {word}{suffix}", 1))
            items.append(src.replace(old, f"{prefix} {digit}{suffix}", 1))
        return items

    for cur in list(out):
        out.extend(add_variants(cur))

    out = [normalize_query_text_loose(x) for x in out if clean_text(x)]
    return dedupe_keep_order(out)

def build_step2_base_queries(addr: str):
    return dedupe_keep_order([normalize_address(addr), simplify_address(addr)])

def build_step2_variant_queries(addr: str):
    base = build_step2_base_queries(addr)
    out = []
    for item in base:
        out.extend(generate_admin_number_variants(item))
        short_item = shorten_region_prefix(item)
        if short_item:
            out.extend(generate_admin_number_variants(short_item))
    out = dedupe_keep_order([clean_text(x) for x in out if clean_text(x)])
    out = [x for x in out if x not in base]
    return out[:8]

def region_hint(addr: str, n=3):
    a = simplify_address(addr)
    if not a:
        return ""
    parts = a.split()
    return " ".join(parts[:min(n, len(parts))]).strip()

def lot_hint(addr: str):
    return extract_lot_token(addr)

def build_name_queries(row):
    name_raw = clean_text(row["이름"])
    name_core = strip_corp_tokens(name_raw)
    brand = normalize_brand(row["상표"])
    brand_aliases = BRAND_ALIASES.get(brand, [brand])

    rg2 = region_hint(row["주소"], n=2)
    rg3 = region_hint(row["주소"], n=3)
    lot = lot_hint(row["주소"])

    queries = []
    name_variants = dedupe_keep_order([name_raw, name_core])

    for nm in name_variants:
        if not nm:
            continue

        queries.append(nm)
        if rg2:
            queries.append(f"{nm} {rg2}")
        if rg3:
            queries.append(f"{nm} {rg3}")

        for brand_alias in brand_aliases[:2]:
            queries.append(f"{brand_alias} {nm}")
            if rg2:
                queries.append(f"{brand_alias} {nm} {rg2}")
            if rg3:
                queries.append(f"{brand_alias} {nm} {rg3}")
            if lot:
                queries.append(f"{brand_alias} {nm} {lot}")

    return dedupe_keep_order([clean_text(x) for x in queries if clean_text(x)])

def normalize_compare_address(addr: str) -> str:
    a = clean_text(addr)
    if not a:
        return ""

    a = re.sub(r"[📍•·▪▶◆◇■□▲△★☆※◎◈☞]", " ", a)
    a = re.sub(r"^\s*대한민국\s*", "", a)
    a = re.sub(r"^\s*한국\s*", "", a)
    a = re.sub(r"^\s*KR\s+", "", a)

    for tok in UI_NOISE_TOKENS + ["주소", "주소 복사"]:
        a = a.replace(tok, " ")

    a = a.replace(",", " ")
    a = a.replace("，", " ")
    a = a.replace("/", " ")
    a = a.replace("(", " ")
    a = a.replace(")", " ")
    a = re.sub(r"\s+", " ", a).strip()

    a = normalize_address(a)
    return clean_text(a)

def simplify_compare_address(addr: str) -> str:
    a = normalize_compare_address(addr)
    if not a:
        return ""

    a = re.sub(r"\b\d+\s*층\b", " ", a, flags=re.I)
    a = re.sub(r"\b\d+(?:-\d+)?\s*호\b", " ", a, flags=re.I)
    a = re.sub(r"\b\d+(?:-\d+)?\s*실\b", " ", a, flags=re.I)
    a = re.sub(r"\b\d+\s*F\b", " ", a, flags=re.I)
    a = re.sub(r"\b[ABCD]\s*동\b", " ", a, flags=re.I)
    a = re.sub(r"\bB\/D\b", " ", a, flags=re.I)
    a = re.sub(r"\b빌딩\b", " ", a)
    a = re.sub(r"\s*외\s*\d+\s*필지.*$", " ", a)
    a = re.sub(r"\s*외\d+\s*필지.*$", " ", a)
    a = re.sub(r"\s*\d+-\d+\s*블록.*$", " ", a)
    a = a.replace("번지", " ")
    a = re.sub(r"\s+", " ", a).strip()
    return a

def _remove_admin_number_tokens_for_lot_scan(text: str) -> str:
    s = clean_text(text)
    s = re.sub(r"(\d+)\s*(동|가|리|통|반)", r" \2", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def get_last_lot_match(addr: str):
    a = simplify_compare_address(addr)
    if not a:
        return None

    a = _remove_admin_number_tokens_for_lot_scan(a)
    matches = list(re.finditer(r"(산\s*\d+(?:-\d+)?|\d+(?:-\d+)?)", a))
    if not matches:
        return None
    return matches[-1]

def extract_lot_token(addr: str) -> str:
    m = get_last_lot_match(addr)
    if not m:
        return ""
    return re.sub(r"\s+", "", m.group(1))

def extract_prefix_before_lot(addr: str) -> str:
    a = simplify_compare_address(addr)
    if not a:
        return ""

    lot_scan_text = _remove_admin_number_tokens_for_lot_scan(a)
    m = get_last_lot_match(addr)
    if not m:
        return ""

    prefix = lot_scan_text[:m.start()].strip()
    prefix = re.sub(r"\s+", " ", prefix).strip()
    return prefix

def compare_candidate_address(candidate_addr: str, raw_addr: str, search_addr: str):
    candidate_raw = clean_text(candidate_addr)
    if not candidate_raw:
        return False, "empty_candidate_address", {
            "candidate_norm": "",
            "candidate_simple": "",
        }

    cand_norm = normalize_compare_address(candidate_raw)
    cand_simple = simplify_compare_address(candidate_raw)

    raw_norm = normalize_compare_address(raw_addr)
    raw_simple = simplify_compare_address(raw_addr)
    search_norm = normalize_compare_address(search_addr)
    search_simple = simplify_compare_address(search_addr)

    targets = dedupe_keep_order([raw_norm, raw_simple, search_norm, search_simple])

    if cand_norm in targets or cand_simple in targets:
        return True, "exact_string_match", {
            "candidate_norm": cand_norm,
            "candidate_simple": cand_simple,
        }

    cand_lot = extract_lot_token(candidate_raw)
    raw_lot = extract_lot_token(raw_addr)
    search_lot = extract_lot_token(search_addr)

    if not cand_lot:
        return False, "candidate_has_no_lot", {
            "candidate_norm": cand_norm,
            "candidate_simple": cand_simple,
        }

    base_lots = [x for x in [raw_lot, search_lot] if x]
    if cand_lot not in base_lots:
        return False, "lot_mismatch", {
            "candidate_norm": cand_norm,
            "candidate_simple": cand_simple,
        }

    cand_prefix = extract_prefix_before_lot(candidate_raw)
    base_prefixes = [x for x in [extract_prefix_before_lot(raw_addr), extract_prefix_before_lot(search_addr)] if x]

    for bp in base_prefixes:
        if bp and (cand_prefix == bp or cand_prefix in bp or bp in cand_prefix):
            return True, "lot_and_prefix_match", {
                "candidate_norm": cand_norm,
                "candidate_simple": cand_simple,
            }

    return False, "no_match", {
        "candidate_norm": cand_norm,
        "candidate_simple": cand_simple,
    }

def address_similarity_score(candidate_addr: str, raw_addr: str, search_addr: str) -> int:
    cand_norm = normalize_compare_address(candidate_addr)
    cand_simple = simplify_compare_address(candidate_addr)

    targets = dedupe_keep_order([
        normalize_compare_address(raw_addr),
        simplify_compare_address(raw_addr),
        normalize_compare_address(search_addr),
        simplify_compare_address(search_addr),
    ])

    if not cand_norm:
        return 0

    base = 0.0
    for t in targets:
        if not t:
            continue
        base = max(base, SequenceMatcher(None, cand_norm, t).ratio())
        if cand_simple:
            base = max(base, SequenceMatcher(None, cand_simple, t).ratio())

    score = int(round(base * 100))

    cand_lot = extract_lot_token(candidate_addr)
    base_lots = [extract_lot_token(raw_addr), extract_lot_token(search_addr)]
    base_lots = [x for x in base_lots if x]

    if not cand_lot:
        score -= 20
    elif cand_lot in base_lots:
        score += 15
    else:
        score -= 10

    cand_prefix = extract_prefix_before_lot(candidate_addr)
    base_prefixes = [extract_prefix_before_lot(raw_addr), extract_prefix_before_lot(search_addr)]
    base_prefixes = [x for x in base_prefixes if x]

    if cand_prefix and any(cand_prefix == bp or cand_prefix in bp or bp in cand_prefix for bp in base_prefixes):
        score += 10

    return max(0, min(score, 100))

def is_same_name_candidate(candidate_name: str, target_name: str) -> bool:
    score, _ = name_match_score(candidate_name, target_name)
    return score >= 58

def coord_key(lat, lng, ndigits=7):
    if pd.isna(lat) or pd.isna(lng):
        return None
    return (round(float(lat), ndigits), round(float(lng), ndigits))

def parse_address_from_text(text: str) -> str:
    text = clean_text(text)
    if not text:
        return ""

    m = re.search(r"(?:주소(?:\s*복사)?\s*[:：]?\s*)(.+)", text)
    if m:
        return clean_text(m.group(1))

    return text

def extract_address_like_strings_from_body(text: str):
    txt = clean_text(text)
    if not txt:
        return []

    prefix = (
        r"(?:서울특별시|부산광역시|대구광역시|인천광역시|광주광역시|대전광역시|"
        r"울산광역시|세종특별자치시|경기도|강원도|충청북도|충청남도|전라북도|"
        r"전라남도|경상북도|경상남도|제주특별자치도|서울|부산|대구|인천|광주|"
        r"대전|울산|세종|경기|강원|충북|충남|전북|전남|경북|경남|제주)"
    )
    lot = r"(?:산\s*\d+(?:-\d+)?|\d+(?:-\d+)?)"
    pattern = re.compile(rf"({prefix}[^\n]{{0,90}}?{lot})")

    matches = []
    for m in pattern.finditer(txt):
        cand = clean_text(m.group(1))
        if 8 <= len(cand) <= 120:
            matches.append(cand)

    return dedupe_keep_order(matches)

def pick_best_address_candidate(candidates, raw_addr, search_addr):
    if not candidates:
        return ""

    scored = []
    for cand in candidates:
        matched, reason, _ = compare_candidate_address(cand, raw_addr, search_addr)
        sim = address_similarity_score(cand, raw_addr, search_addr)
        score = 0
        if reason == "exact_string_match":
            score += 200
        elif reason == "lot_and_prefix_match":
            score += 150
        score += sim
        scored.append((score, -len(cand), cand))

    scored.sort(reverse=True)
    return scored[0][2]

def extract_exact_coords_from_url(url: str):
    m = re.search(r"!3d(-?\d+(?:\.\d+)?)!4d(-?\d+(?:\.\d+)?)", url)
    if not m:
        return None
    lat = float(m.group(1))
    lng = float(m.group(2))
    return {"lat": lat, "lng": lng, "source": "url_3d4d"}

def extract_center_coords_from_url(url: str):
    m = re.search(r"@(-?\d+(?:\.\d+)?),(-?\d+(?:\.\d+)?),", url)
    if not m:
        return None
    lat = float(m.group(1))
    lng = float(m.group(2))
    return {"lat": lat, "lng": lng, "source": "url_center"}

def extract_coords_from_text(text: str):
    for m in re.finditer(r"(?<!\d)(-?\d{1,3}\.\d{4,}),\s*(-?\d{1,3}\.\d{4,})(?!\d)", text):
        lat = float(m.group(1))
        lng = float(m.group(2))
        if -90 <= lat <= 90 and -180 <= lng <= 180:
            return {"lat": lat, "lng": lng, "source": "share_dialog_text"}
    return None

async def dismiss_google_consent(page):
    selectors = [
        'button:has-text("모두 수락")',
        'button:has-text("Accept all")',
        '[aria-label="모두 수락"]',
        '[aria-label="Accept all"]',
    ]
    for sel in selectors:
        try:
            loc = page.locator(sel)
            if await loc.count() > 0:
                await loc.first.click(timeout=2500)
                await page.wait_for_timeout(1200)
                return True
        except Exception:
            pass
    return False

async def detect_block(page):
    try:
        txt = (await page.locator("body").inner_text(timeout=2000)).lower()
    except Exception:
        return False
    signals = ["unusual traffic", "자동화된 트래픽", "sorry", "죄송합니다"]
    return any(s in txt for s in signals)

async def has_share_button(page):
    selectors = [
        'button[aria-label="공유"]',
        'button[aria-label="Share"]',
        'button[data-value="공유"]',
    ]
    for sel in selectors:
        try:
            loc = page.locator(sel)
            if await loc.count() > 0 and await loc.first.is_visible():
                return True
        except Exception:
            pass
    return False

async def open_share_dialog(page):
    selectors = [
        'button[aria-label="공유"]',
        'button[aria-label="Share"]',
        'button[data-value="공유"]',
    ]
    for sel in selectors:
        try:
            loc = page.locator(sel)
            if await loc.count() > 0:
                await loc.first.click(timeout=3000)
                await page.wait_for_timeout(1200)
                return True
        except Exception:
            pass
    return False

async def close_dialog(page):
    for sel in ['button[aria-label="닫기"]', 'button[aria-label="Close"]']:
        try:
            loc = page.locator(sel)
            if await loc.count() > 0:
                await loc.first.click(timeout=1000)
                await page.wait_for_timeout(500)
                return
        except Exception:
            pass
    try:
        await page.keyboard.press("Escape")
        await page.wait_for_timeout(500)
    except Exception:
        pass

async def click_first_search_result(page):
    selectors = ['a[href*="/maps/place/"]', 'a.hfpxzc']
    for sel in selectors:
        try:
            loc = page.locator(sel)
            count = min(await loc.count(), 5)
            for i in range(count):
                item = loc.nth(i)
                href = await item.get_attribute("href") or ""
                if "/maps/place/" in href:
                    await item.click(timeout=3000)
                    await page.wait_for_timeout(1800)
                    return True
        except Exception:
            pass
    return False

def build_search_url(query: str) -> str:
    return f"https://www.google.com/maps/search/?api=1&hl=ko&query={quote(query)}"

async def geocode_fast_original(page, raw_query: str, max_wait_loops: int = 12):
    tried = set()

    for query in generate_query_candidates_original(raw_query):
        if query in tried:
            continue
        tried.add(query)

        await page.goto(build_search_url(query), wait_until="domcontentloaded", timeout=30000)
        await dismiss_google_consent(page)
        await page.wait_for_timeout(2200)

        if await detect_block(page):
            raise RuntimeError("Google Maps 자동화 차단 페이지 감지")

        for _ in range(max_wait_loops):
            exact = extract_exact_coords_from_url(page.url)
            if exact:
                exact["query"] = query
                exact["url"] = page.url
                return exact

            if not await has_share_button(page):
                clicked = await click_first_search_result(page)
                if clicked:
                    exact = extract_exact_coords_from_url(page.url)
                    if exact:
                        exact["query"] = query
                        exact["url"] = page.url
                        return exact

            await page.wait_for_timeout(500)

        if await open_share_dialog(page):
            try:
                body_text = await page.locator("body").inner_text(timeout=3000)
                result = extract_coords_from_text(body_text)
                if result:
                    result["query"] = query
                    result["url"] = page.url
                    return result
            finally:
                await close_dialog(page)

    return None

async def create_browser_context(p, headless=True):
    browser = await p.chromium.launch(
        headless=headless,
        args=[
            "--no-sandbox",
            "--disable-dev-shm-usage",
            "--disable-blink-features=AutomationControlled",
        ],
    )

    context = await browser.new_context(
        locale="ko-KR",
        viewport={"width": 1440, "height": 1100},
        user_agent=(
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/131.0.0.0 Safari/537.36"
        ),
        extra_http_headers={"accept-language": "ko-KR,ko;q=0.9,en-US;q=0.8"},
    )
    context.set_default_timeout(15000)

    await context.add_init_script("""
        Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
    """)

    page = await context.new_page()
    await page.goto("https://www.google.com/maps?hl=ko", wait_until="domcontentloaded", timeout=30000)
    await dismiss_google_consent(page)
    await page.wait_for_timeout(1500)

    return browser, context, page

async def page_has_detail_address(page):
    selectors = [
        'button[data-item-id="address"]',
        'button[data-item-id*="address"]',
        '[data-item-id="address"]',
        '[data-item-id*="address"]',
        'button[aria-label^="주소"]',
        'button[aria-label*="주소 복사"]',
    ]
    for sel in selectors:
        try:
            loc = page.locator(sel)
            if await loc.count() > 0:
                return True
        except Exception:
            pass
    return False

async def read_place_detail(page, raw_addr=None, search_addr=None, allow_center_fallback=False):
    detail = {
        "place_name": "",
        "detail_address": "",
        "lat": np.nan,
        "lng": np.nan,
        "coord_source": "",
        "url": page.url,
    }

    for sel in ["h1", "h1 span"]:
        try:
            loc = page.locator(sel)
            if await loc.count() > 0:
                txt = clean_text(await loc.first.inner_text())
                if txt:
                    detail["place_name"] = txt
                    break
        except Exception:
            pass

    address_candidates = []

    address_selectors = [
        'button[data-item-id="address"]',
        'button[data-item-id*="address"]',
        '[data-item-id="address"]',
        '[data-item-id*="address"]',
        'button[aria-label^="주소"]',
        'button[aria-label*="주소 복사"]',
    ]

    for sel in address_selectors:
        try:
            loc = page.locator(sel)
            cnt = min(await loc.count(), 8)
            for i in range(cnt):
                item = loc.nth(i)
                txt = ""
                aria = ""

                try:
                    txt = clean_text(await item.inner_text())
                except Exception:
                    pass

                try:
                    aria = clean_text(await item.get_attribute("aria-label"))
                except Exception:
                    pass

                for cand in [txt, aria]:
                    cand = parse_address_from_text(cand)
                    cand = clean_text(cand)
                    if cand and cand not in address_candidates:
                        address_candidates.append(cand)
        except Exception:
            pass

    if not address_candidates:
        try:
            body_text = await page.locator("body").inner_text(timeout=3000)
            address_candidates.extend(extract_address_like_strings_from_body(body_text))
        except Exception:
            pass

    address_candidates = dedupe_keep_order(address_candidates)
    if address_candidates:
        if raw_addr or search_addr:
            detail["detail_address"] = pick_best_address_candidate(
                address_candidates,
                raw_addr or "",
                search_addr or "",
            )
        else:
            detail["detail_address"] = address_candidates[0]

    exact = extract_exact_coords_from_url(page.url)
    if exact:
        detail["lat"] = exact["lat"]
        detail["lng"] = exact["lng"]
        detail["coord_source"] = exact["source"]
        return detail

    if await open_share_dialog(page):
        try:
            body_text = await page.locator("body").inner_text(timeout=3000)
            shared = extract_coords_from_text(body_text)
            if shared:
                detail["lat"] = shared["lat"]
                detail["lng"] = shared["lng"]
                detail["coord_source"] = shared["source"]
                return detail
        except Exception:
            pass
        finally:
            await close_dialog(page)

    if allow_center_fallback:
        center = extract_center_coords_from_url(page.url)
        if center:
            detail["lat"] = center["lat"]
            detail["lng"] = center["lng"]
            detail["coord_source"] = center["source"]

    return detail

async def get_search_candidates(page, max_results=8):
    candidates = []

    if "/maps/place/" in page.url or "/place/" in page.url:
        candidates.append({"label": "", "href": page.url, "kind": "direct_detail"})
        return candidates

    try:
        if await page_has_detail_address(page):
            candidates.append({"label": "", "href": page.url, "kind": "direct_detail"})
            return candidates
    except Exception:
        pass

    seen = set()
    for sel in ['a[href*="/maps/place/"]', 'a.hfpxzc']:
        try:
            loc = page.locator(sel)
            cnt = await loc.count()

            for i in range(cnt):
                item = loc.nth(i)

                try:
                    href = clean_text(await item.get_attribute("href"))
                except Exception:
                    href = ""

                if not href or "/maps/place/" not in href or href in seen:
                    continue
                seen.add(href)

                label = ""
                for getter in [
                    lambda: item.get_attribute("aria-label"),
                    lambda: item.inner_text(),
                    lambda: item.text_content(),
                ]:
                    try:
                        v = clean_text(await getter())
                        if v:
                            label = v
                            break
                    except Exception:
                        pass

                candidates.append({
                    "label": label,
                    "href": href,
                    "kind": "search_list",
                })

                if len(candidates) >= max_results:
                    return candidates
        except Exception:
            pass

    return candidates

async def collect_candidates_from_queries(page, query_items, max_results_per_query=8, max_unique_candidates=12):
    ref_map = {}

    for qtype, query in query_items:
        if not query:
            continue

        try:
            await page.goto(build_search_url(query), wait_until="domcontentloaded", timeout=30000)
            await dismiss_google_consent(page)
            await page.wait_for_timeout(1800)

            if await detect_block(page):
                raise RuntimeError("Google Maps 자동화 차단 페이지 감지")

            candidates = await get_search_candidates(page, max_results=max_results_per_query)

            for cand in candidates:
                href = cand["href"]
                if href not in ref_map:
                    ref_map[href] = {
                        "href": href,
                        "labels": [],
                        "queries": [],
                        "query_types": [],
                        "kind": cand["kind"],
                    }

                if cand.get("label"):
                    ref_map[href]["labels"].append(cand["label"])
                ref_map[href]["queries"].append(query)
                ref_map[href]["query_types"].append(qtype)

                if len(ref_map) >= max_unique_candidates:
                    break

            if len(ref_map) >= max_unique_candidates:
                break

        except Exception as e:
            print(f"[WARN] query 실패 | type={qtype} | query={query} | {type(e).__name__}: {e}")

        await page.wait_for_timeout(int(random.uniform(300, 800)))

    return list(ref_map.values())

async def inspect_candidate_details(page, row, candidate_refs, allow_center_fallback=False):
    evaluated = []

    for idx, ref in enumerate(candidate_refs, start=1):
        try:
            await page.goto(ref["href"], wait_until="domcontentloaded", timeout=30000)
            await dismiss_google_consent(page)
            await page.wait_for_timeout(1400)

            detail = await read_place_detail(
                page,
                raw_addr=row["주소"],
                search_addr=row["검색주소"],
                allow_center_fallback=allow_center_fallback,
            )

            candidate_name = clean_text(detail["place_name"] or " ".join(dedupe_keep_order(ref.get("labels", []))))
            candidate_address = clean_text(detail["detail_address"])

            address_matched, address_reason, _ = compare_candidate_address(
                candidate_address,
                row["주소"],
                row["검색주소"],
            )

            addr_sim = address_similarity_score(candidate_address, row["주소"], row["검색주소"])
            nm_score, nm_reason = name_match_score(candidate_name, row["이름"])
            br_score, br_reason = brand_match_score(candidate_name, row["상표"])

            evaluated.append({
                "candidate_name": candidate_name,
                "candidate_address": candidate_address,
                "lat": detail["lat"],
                "lng": detail["lng"],
                "coord_source": detail["coord_source"],
                "url": detail["url"],
                "href": ref["href"],
                "labels": dedupe_keep_order(ref.get("labels", [])),
                "queries": dedupe_keep_order(ref.get("queries", [])),
                "query_types": dedupe_keep_order(ref.get("query_types", [])),
                "address_matched": address_matched,
                "address_reason": address_reason,
                "address_similarity": addr_sim,
                "name_score": nm_score,
                "name_reason": nm_reason,
                "brand_score": br_score,
                "brand_reason": br_reason,
            })

        except Exception as e:
            print(f"[WARN] 후보 평가 실패 | {type(e).__name__}: {e}")

        await page.wait_for_timeout(int(random.uniform(250, 650)))

    return evaluated

def select_step2_candidate(evaluated, target_name):
    address_matches = [x for x in evaluated if x["address_matched"] and coord_key(x["lat"], x["lng"]) is not None]

    if len(address_matches) == 0:
        return None, "no_address_match"

    if len(address_matches) == 1:
        return address_matches[0], "one_address_match"

    coord_set = {coord_key(x["lat"], x["lng"]) for x in address_matches}
    if len(coord_set) == 1:
        return address_matches[0], "multi_address_same_coord"

    same_name = [x for x in address_matches if is_same_name_candidate(x["candidate_name"], target_name)]

    if len(same_name) == 1:
        return same_name[0], "multi_address_choose_same_name"

    if len(same_name) >= 2:
        coord_set2 = {coord_key(x["lat"], x["lng"]) for x in same_name}
        if len(coord_set2) == 1:
            return same_name[0], "multi_address_same_name_same_coord"

    return None, "multi_address_unresolved"

def select_step3_candidate(evaluated, row):
    if not evaluated:
        return None, "no_name_candidates"

    strong_name = [x for x in evaluated if x["name_score"] >= 40]

    if not strong_name:
        return None, "no_strong_name_candidates"

    cand, reason = select_step2_candidate(strong_name, row["이름"])
    if cand is not None:
        return cand, f"name_search_{reason}"

    strong_name.sort(
        key=lambda x: (
            x["address_similarity"],
            x["name_score"],
            x["brand_score"],
        ),
        reverse=True,
    )

    top1 = strong_name[0]
    top2 = strong_name[1] if len(strong_name) >= 2 else None

    enough_similarity = top1["address_similarity"] >= 75
    enough_name = top1["name_score"] >= 45

    if enough_similarity and enough_name:
        if top2 is None:
            return top1, "name_search_best_similar"

        if (top1["address_similarity"] - top2["address_similarity"] >= 10) or (top1["name_score"] - top2["name_score"] >= 15):
            return top1, "name_search_best_similar"

    return None, "name_search_ambiguous"

def save_progress(df, output_file):
    save_cols = [
        "상표", "대상", "이름", "주소",
        "경도", "위도",
        "결과유형", "판정사유",
        "선택후보명", "선택상세주소", "선택쿼리", "선택URL",
    ]
    out = df.copy()
    for col in save_cols:
        if col not in out.columns:
            out[col] = np.nan if col in ("경도", "위도") else ""
    out[save_cols].to_csv(output_file, index=False, encoding="utf-8-sig")


# =========================================================
# 3) 사용자설정 및 실행
# =========================================================
print("[3/3] 사용자 설정 및 실행")

HEADLESS = True
USE_RESUME = False
ALLOW_CENTER_FALLBACK = False

SAVE_EVERY = 20
RESTART_EVERY = 120
MAX_RESULTS_PER_QUERY = 8
MAX_UNIQUE_CANDIDATES = 12

NAME_SEARCH_MAX_RESULTS_PER_QUERY = 6
NAME_SEARCH_MAX_UNIQUE_CANDIDATES = 10

if "DATA_PATH" in globals():
    INPUT_FILE = os.path.join(DATA_PATH, "facility_data.csv")
else:
    INPUT_FILE = "/mnt/data/facility_data.csv"

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"입력 파일이 없습니다: {INPUT_FILE}")

if "PROCESSED_PATH" in globals():
    OUTPUT_DIR = os.path.join(PROCESSED_PATH, "additional_data")
else:
    OUTPUT_DIR = os.path.dirname(INPUT_FILE)

os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "1 facility_location_data.csv")

# 필요 없으면 비우면 됨
MANUAL_QUERY_OVERRIDES = {
    normalize_address("제주 제주시 용담 2동 2643-1"): [
        "제주 제주시 용담 이동 2643-1",
        "제주특별자치도 제주시 용담이동 2643-1",
    ],
}

# 필요 없으면 비우면 됨
MANUAL_COORD_OVERRIDES_BY_ADDR = {
    normalize_address("제주 제주시 용담 2동 2643-1"): {
        "lat": 33.500531,
        "lng": 126.511275,
        "reason": "user_verified_manual_coord",
    },
    normalize_address("경기도 부천시 오정구 고강동 218-6"): {
        "lat": 37.534301,
        "lng": 126.816645,
        "reason": "user_verified_manual_coord",
    },
}

print(f" - INPUT_FILE : {INPUT_FILE}")
print(f" - OUTPUT_FILE: {OUTPUT_FILE}")

source_df = pd.read_csv(INPUT_FILE)
required_cols = {"상표", "구분", "이름", "주소"}
missing_cols = required_cols - set(source_df.columns)
if missing_cols:
    raise ValueError(f"facility_data.csv 필수 컬럼 누락: {missing_cols}")

source_df = source_df[["상표", "구분", "이름", "주소"]].copy()
source_df = source_df.rename(columns={"구분": "대상"})
source_df["상표"] = source_df["상표"].apply(normalize_brand)
source_df["대상"] = source_df["대상"].apply(normalize_target)
source_df["이름"] = source_df["이름"].apply(clean_text)
source_df["주소"] = source_df["주소"].apply(clean_text)
source_df["검색주소"] = source_df["주소"].apply(normalize_address)
source_df["간소주소"] = source_df["주소"].apply(simplify_address)

result_df = source_df.copy()
for col in ["경도", "위도", "결과유형", "판정사유", "선택후보명", "선택상세주소", "선택쿼리", "선택URL"]:
    result_df[col] = np.nan if col in ("경도", "위도") else ""

if USE_RESUME and os.path.exists(OUTPUT_FILE):
    old = pd.read_csv(OUTPUT_FILE)
    keep_cols = ["상표", "대상", "이름", "주소", "경도", "위도", "결과유형", "판정사유", "선택후보명", "선택상세주소", "선택쿼리", "선택URL"]
    if set(["상표", "대상", "이름", "주소", "경도", "위도"]).issubset(old.columns):
        for col in keep_cols:
            if col not in old.columns:
                old[col] = np.nan if col in ("경도", "위도") else ""
        old = old[keep_cols].copy()
        result_df = result_df.drop(columns=["경도", "위도", "결과유형", "판정사유", "선택후보명", "선택상세주소", "선택쿼리", "선택URL"]).merge(
            old,
            on=["상표", "대상", "이름", "주소"],
            how="left",
        )
        print(" - 기존 결과 이어받기 활성화")

address_cache = {}
done_mask = result_df["경도"].notna() & result_df["위도"].notna()
for _, row in result_df.loc[done_mask].iterrows():
    norm_addr = normalize_address(row["주소"])
    address_cache[norm_addr] = {
        "lat": float(row["위도"]),
        "lng": float(row["경도"]),
        "candidate_name": clean_text(row.get("선택후보명", "")),
        "candidate_address": clean_text(row.get("선택상세주소", "")),
        "queries": [clean_text(row.get("선택쿼리", ""))] if clean_text(row.get("선택쿼리", "")) else [],
        "url": clean_text(row.get("선택URL", "")),
        "reason": clean_text(row.get("판정사유", "")),
    }

async def main():
    global result_df, address_cache

    pending_idx = result_df[result_df["경도"].isna() | result_df["위도"].isna()].index.tolist()

    print(f"총 행 수: {len(result_df)}")
    print(f"처리 대상 행 수: {len(pending_idx)}")
    print(f"유니크 검색주소 수: {result_df['검색주소'].nunique()}")
    print(f"기존 캐시 수: {len(address_cache)}")

    save_progress(result_df, OUTPUT_FILE)

    if not pending_idx:
        return result_df

    async with async_playwright() as p:
        browser = None
        context = None
        page = None

        try:
            browser, context, page = await create_browser_context(p, headless=HEADLESS)

            for order, idx in enumerate(tqdm(pending_idx, desc="enhanced geocoding"), start=1):
                if order != 1 and (order - 1) % RESTART_EVERY == 0:
                    print(f"\n[브라우저 재시작] {order-1}건 처리 후 재시작")
                    try:
                        await context.close()
                    except Exception:
                        pass
                    try:
                        await browser.close()
                    except Exception:
                        pass
                    browser, context, page = await create_browser_context(p, headless=HEADLESS)

                row = result_df.loc[idx, ["상표", "대상", "이름", "주소", "검색주소", "간소주소"]].to_dict()
                norm_addr = row["검색주소"]

                print("\n" + "=" * 140)
                print(f"[{order}/{len(pending_idx)}] 상표={row['상표']} | 대상={row['대상']} | 이름={row['이름']}")
                print(f"원본주소 : {row['주소']}")
                print(f"검색주소 : {row['검색주소']}")

                # 0) 수동 좌표 오버라이드
                if norm_addr in MANUAL_COORD_OVERRIDES_BY_ADDR:
                    manual = MANUAL_COORD_OVERRIDES_BY_ADDR[norm_addr]
                    result_df.loc[idx, "경도"] = manual["lng"]
                    result_df.loc[idx, "위도"] = manual["lat"]
                    result_df.loc[idx, "결과유형"] = "MANUAL_COORD"
                    result_df.loc[idx, "판정사유"] = manual.get("reason", "manual_coord")
                    result_df.loc[idx, "선택후보명"] = ""
                    result_df.loc[idx, "선택상세주소"] = norm_addr
                    result_df.loc[idx, "선택쿼리"] = ""
                    result_df.loc[idx, "선택URL"] = ""
                    address_cache[norm_addr] = {
                        "lat": float(manual["lat"]),
                        "lng": float(manual["lng"]),
                        "candidate_name": "",
                        "candidate_address": norm_addr,
                        "queries": [],
                        "url": "",
                        "reason": manual.get("reason", "manual_coord"),
                    }
                    print(f" -> 수동 좌표 적용: {manual['lat']}, {manual['lng']}")
                    continue

                # 0.5) 캐시
                if norm_addr in address_cache:
                    cached = address_cache[norm_addr]
                    result_df.loc[idx, "경도"] = cached["lng"]
                    result_df.loc[idx, "위도"] = cached["lat"]
                    result_df.loc[idx, "결과유형"] = "CACHE"
                    result_df.loc[idx, "판정사유"] = cached.get("reason", "cache")
                    result_df.loc[idx, "선택후보명"] = cached.get("candidate_name", "")
                    result_df.loc[idx, "선택상세주소"] = cached.get("candidate_address", "")
                    result_df.loc[idx, "선택쿼리"] = " || ".join(cached.get("queries", []))
                    result_df.loc[idx, "선택URL"] = cached.get("url", "")
                    print(f" -> 캐시 적용: {cached['lat']}, {cached['lng']}")
                    continue

                # 1) 기존 1번 파일 방식
                fast_result = None
                try:
                    fast_result = await geocode_fast_original(page, row["주소"])
                except Exception as e:
                    print(f"[WARN] 1차 빠른 조회 실패: {type(e).__name__}: {e}")

                if fast_result is not None:
                    result_df.loc[idx, "경도"] = fast_result["lng"]
                    result_df.loc[idx, "위도"] = fast_result["lat"]
                    result_df.loc[idx, "결과유형"] = "FAST_ORIGINAL"
                    result_df.loc[idx, "판정사유"] = fast_result.get("source", "fast_original")
                    result_df.loc[idx, "선택후보명"] = ""
                    result_df.loc[idx, "선택상세주소"] = row["검색주소"]
                    result_df.loc[idx, "선택쿼리"] = fast_result.get("query", "")
                    result_df.loc[idx, "선택URL"] = fast_result.get("url", "")

                    address_cache[norm_addr] = {
                        "lat": float(fast_result["lat"]),
                        "lng": float(fast_result["lng"]),
                        "candidate_name": "",
                        "candidate_address": row["검색주소"],
                        "queries": [fast_result.get("query", "")] if fast_result.get("query", "") else [],
                        "url": fast_result.get("url", ""),
                        "reason": fast_result.get("source", "fast_original"),
                    }

                    print(f" -> 1차 성공: {fast_result['lat']}, {fast_result['lng']} | {fast_result.get('query', '')}")
                    continue

                # 2) 목록 기반 주소 판정 - 기본 주소
                base_queries = [("address", q) for q in build_step2_base_queries(row["주소"])]
                base_refs = await collect_candidates_from_queries(
                    page,
                    base_queries,
                    max_results_per_query=MAX_RESULTS_PER_QUERY,
                    max_unique_candidates=MAX_UNIQUE_CANDIDATES,
                )

                chosen = None
                chosen_reason = ""

                if len(base_refs) > 0:
                    base_eval = await inspect_candidate_details(
                        page,
                        row,
                        base_refs,
                        allow_center_fallback=ALLOW_CENTER_FALLBACK,
                    )
                    chosen, chosen_reason = select_step2_candidate(base_eval, row["이름"])

                    print(f" -> 2차 기본주소 평가 후보 수: {len(base_eval)}")
                    print(f" -> 2차 기본주소 판정: {chosen_reason}")

                    if chosen is None and chosen_reason == "no_address_match":
                        # 주소 일치 없을 때만 숫자/한글 변형 주소 재시도
                        variant_queries = [("variant_address", q) for q in build_step2_variant_queries(row["주소"])]
                        manual_queries = [("variant_address", q) for q in MANUAL_QUERY_OVERRIDES.get(norm_addr, [])]
                        variant_refs = await collect_candidates_from_queries(
                            page,
                            manual_queries + variant_queries,
                            max_results_per_query=MAX_RESULTS_PER_QUERY,
                            max_unique_candidates=MAX_UNIQUE_CANDIDATES,
                        )

                        if len(variant_refs) > 0:
                            variant_eval = await inspect_candidate_details(
                                page,
                                row,
                                variant_refs,
                                allow_center_fallback=ALLOW_CENTER_FALLBACK,
                            )
                            chosen, chosen_reason = select_step2_candidate(variant_eval, row["이름"])
                            print(f" -> 2차 변형주소 평가 후보 수: {len(variant_eval)}")
                            print(f" -> 2차 변형주소 판정: {chosen_reason}")
                else:
                    chosen_reason = "no_search_list"

                if chosen is not None:
                    result_df.loc[idx, "경도"] = chosen["lng"]
                    result_df.loc[idx, "위도"] = chosen["lat"]
                    result_df.loc[idx, "결과유형"] = f"LIST_ADDRESS::{chosen_reason}"
                    result_df.loc[idx, "판정사유"] = chosen["address_reason"]
                    result_df.loc[idx, "선택후보명"] = chosen["candidate_name"]
                    result_df.loc[idx, "선택상세주소"] = chosen["candidate_address"]
                    result_df.loc[idx, "선택쿼리"] = " || ".join(chosen["queries"])
                    result_df.loc[idx, "선택URL"] = chosen["url"]

                    address_cache[norm_addr] = {
                        "lat": float(chosen["lat"]),
                        "lng": float(chosen["lng"]),
                        "candidate_name": chosen["candidate_name"],
                        "candidate_address": chosen["candidate_address"],
                        "queries": chosen["queries"],
                        "url": chosen["url"],
                        "reason": chosen["address_reason"],
                    }

                    print(f" -> 2차 채택: {chosen['candidate_name']} | {chosen['candidate_address']} | {chosen['lat']}, {chosen['lng']}")
                    continue

                # 3) 이름 검색
                name_queries = [("name", q) for q in build_name_queries(row)]
                name_refs = await collect_candidates_from_queries(
                    page,
                    name_queries,
                    max_results_per_query=NAME_SEARCH_MAX_RESULTS_PER_QUERY,
                    max_unique_candidates=NAME_SEARCH_MAX_UNIQUE_CANDIDATES,
                )

                chosen3 = None
                reason3 = "no_name_candidates"

                if len(name_refs) > 0:
                    name_eval = await inspect_candidate_details(
                        page,
                        row,
                        name_refs,
                        allow_center_fallback=ALLOW_CENTER_FALLBACK,
                    )
                    chosen3, reason3 = select_step3_candidate(name_eval, row)
                    print(f" -> 3차 이름검색 평가 후보 수: {len(name_eval)}")
                    print(f" -> 3차 이름검색 판정: {reason3}")

                if chosen3 is not None:
                    result_df.loc[idx, "경도"] = chosen3["lng"]
                    result_df.loc[idx, "위도"] = chosen3["lat"]
                    result_df.loc[idx, "결과유형"] = f"NAME_SEARCH::{reason3}"
                    result_df.loc[idx, "판정사유"] = chosen3["address_reason"]
                    result_df.loc[idx, "선택후보명"] = chosen3["candidate_name"]
                    result_df.loc[idx, "선택상세주소"] = chosen3["candidate_address"]
                    result_df.loc[idx, "선택쿼리"] = " || ".join(chosen3["queries"])
                    result_df.loc[idx, "선택URL"] = chosen3["url"]

                    if chosen3["address_reason"] in ("exact_string_match", "lot_and_prefix_match"):
                        address_cache[norm_addr] = {
                            "lat": float(chosen3["lat"]),
                            "lng": float(chosen3["lng"]),
                            "candidate_name": chosen3["candidate_name"],
                            "candidate_address": chosen3["candidate_address"],
                            "queries": chosen3["queries"],
                            "url": chosen3["url"],
                            "reason": chosen3["address_reason"],
                        }

                    print(f" -> 3차 채택: {chosen3['candidate_name']} | {chosen3['candidate_address']} | {chosen3['lat']}, {chosen3['lng']}")
                else:
                    result_df.loc[idx, "결과유형"] = "FAIL"
                    result_df.loc[idx, "판정사유"] = reason3 if reason3 else chosen_reason
                    print(f" -> 최종 FAIL: {reason3 if reason3 else chosen_reason}")

                if order % SAVE_EVERY == 0 or order == len(pending_idx):
                    save_progress(result_df, OUTPUT_FILE)
                    done_cnt = int((result_df["경도"].notna() & result_df["위도"].notna()).sum())
                    print(f"\n[중간저장] {order}/{len(pending_idx)} | 누적완료 {done_cnt}/{len(result_df)} | {OUTPUT_FILE}")

                await page.wait_for_timeout(int(random.uniform(600, 1400)))

        finally:
            try:
                if context is not None:
                    await context.close()
            except Exception:
                pass
            try:
                if browser is not None:
                    await browser.close()
            except Exception:
                pass

    return result_df

loop = asyncio.get_event_loop()
result_df = loop.run_until_complete(main())

save_progress(result_df, OUTPUT_FILE)

not_nan_df = result_df[result_df["경도"].notna() & result_df["위도"].notna()].copy().reset_index(drop=True)
nan_df = result_df[result_df["경도"].isna() | result_df["위도"].isna()].copy().reset_index(drop=True)

print("\n[완료]")
print(f"저장 경로: {OUTPUT_FILE}")
print(f"not_nan_df 행 수: {len(not_nan_df)}")
print(f"nan_df 행 수    : {len(nan_df)}")
print(result_df.head())

### 1-3. 수동 좌표 보정 및 최종 파일 생성

In [ ]:
stations = {
    "(주)당진엘피지": {"lat": 36.9038445, "lng": 126.6402232},
    "(주)공항석유 가평휴게서(춘천방향)": {"lat": 37.7018696, "lng": 127.5460287},
    "(주)공항석유 가평휴게소(서울방향)": {"lat": 37.7125368, "lng": 127.5498785},
    "광신개발(주)직영우성LPG로트": {"lat": 35.4448587, "lng": 128.5215941},
    "문화가스": {"lat": 35.1266048, "lng": 129.0472887},
    "서곡충전소": {"lat": 35.8379122, "lng": 127.10351},
    "영일충전소": {"lat": 36.86553, "lng": 126.611386},
    "일산충전소": {"lat": 37.6772014, "lng": 126.800009},
    "점보가스충전소": {"lat": 37.4859224, "lng": 126.680055},
    "포항충전소": {"lat": 36.0108221, "lng": 129.3460927},
    "한국광유(주)제2항구주유소": {"lat": 36.050429, "lng": 129.3723946},
    "(주)녹색에너지": {"lat": 35.1306433, "lng": 129.0547559},
    "(주)소모석유": {"lat": 37.314744, "lng": 127.108830},
    "(주)티엔에프하이웨이충전소": {"lat": 35.5902144, "lng": 128.8869702},
    "강구LPG충전소": {"lat": 36.349143, "lng": 129.379108},
    "강동엘피지충전소": {"lat": 37.5034375, "lng": 127.0869375},
    "대성산업 곤지암충전소": {"lat": 37.3535625, "lng": 127.3336875},
    "대성산업 의정부충전소": {"lat": 37.7464033, "lng": 127.0476243},
    "대성산업(주)부산대성충전소": {"lat": 35.1750068, "lng": 128.9840563},
    "동산LPG충전소": {"lat": 35.862988, "lng": 127.0865684},
    "마라톤가스 주식회사": {"lat": 35.0846751, "lng": 128.9801253},
    "반월성가스충전소": {"lat": 37.111473, "lng": 127.550222},
    "발안LPG충전소(주)대야에너지": {"lat": 37.133722, "lng": 126.8962431},
    "상일충전소": {"lat": 37.545091, "lng": 127.1698272},
    "송리원충전소": {"lat": 36.7662467, "lng": 128.6792728},
    "주식회사 학교산업지점": {"lat": 35.0239589, "lng": 126.5373049},
    "중도가스 천안충전소": {"lat": 36.8354881, "lng": 127.1524174},
    "청북가스충전소": {"lat": 37.0316827, "lng": 126.9385716},
    "(주)대하상사": {"lat": 35.8981544, "lng": 128.5756408},
    "(주)동 남 유 화": {"lat": 37.5172363, "lng": 127.0473248},
    "(주)미래아스팔트": {"lat": 37.5175312, "lng": 127.103575},
    "(주)세종코퍼레이션": {"lat": 37.521799, "lng": 126.9281524},
}

In [ ]:
df = pd.read_csv(PROCESSED_PATH + 'additional_data/1 facility_location_data.csv')

remove_list = ['SK네트웍스', '(주)청담산업', '(주)서호유류판매 동행충전소', '개인택시복지충전소']

df = df.loc[~df['이름'].isin(remove_list)]

for name, coord in stations.items():
    df.loc[df['이름'] == name, '경도'] = coord['lng']
    df.loc[df['이름'] == name, '위도'] = coord['lat']
df = df[['상표', '대상', '경도', '위도']]

print(df)

df.to_csv(PROCESSED_PATH + 'additional_data/1 facility_location_data_final.csv', index=False)

In [ ]:
df

## 2. 전국 개별 주유소 데이터

오피넷 가격 다운로드 페이지에서 지역별 CSV를 수집하고, 지역별 가격 시계열과 메타데이터를 만든 뒤 좌표를 보강합니다.

### 2-1. 오피넷 지역별 CSV 다운로드 환경 설정

In [ ]:
# =========================
# 1셀. 환경 설정
# =========================

from google.colab import drive
drive.mount('/content/drive')

!apt-get update -y
!apt-get remove -y chromium-browser chromium-chromedriver || true
!rm -f /usr/bin/chromedriver /usr/bin/chromium-browser || true

!apt-get install -y wget gnupg unzip curl
!mkdir -p /etc/apt/keyrings
!curl -fsSL https://dl.google.com/linux/linux_signing_key.pub | gpg --dearmor -o /etc/apt/keyrings/google-chrome.gpg
!echo "deb [arch=amd64 signed-by=/etc/apt/keyrings/google-chrome.gpg] http://dl.google.com/linux/chrome/deb/ stable main" | tee /etc/apt/sources.list.d/google-chrome.list > /dev/null

!apt-get update -y
!apt-get install -y google-chrome-stable

!pip -q install selenium pandas numpy

import os
import re
import time
import glob
from pathlib import Path

import pandas as pd
import numpy as np

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.chrome.options import Options

ROOT_PATH = "/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/"
DATA_PATH = ROOT_PATH + "data/"
PROCESSED_PATH = ROOT_PATH + "preprocessed_data/"

REGION_PRICE_ROOT = os.path.join(DATA_PATH, "gas_station_prices_by_region")
os.makedirs(REGION_PRICE_ROOT, exist_ok=True)

print("ROOT_PATH:", ROOT_PATH)
print("DATA_PATH:", DATA_PATH)
print("REGION_PRICE_ROOT:", REGION_PRICE_ROOT)

### 2-2. 다운로드 함수 정의

In [ ]:
# =========================
# 2셀. 함수 정의
# =========================

URL = "https://www.opinet.co.kr/user/opdown/opDownload.do"

def sanitize_filename(name):
    return re.sub(r'[\\/:*?"<>|]+', "_", name).strip()

def make_ranges(start_year=2008, end_year=2026):
    ranges = []
    for y in range(start_year, end_year + 1):
        if y == 2008:
            ranges.append((str(y), "20080415", "20081231"))
        else:
            ranges.append((str(y), f"{y}0101", f"{y}1231"))
    return ranges

def current_files(download_dir):
    return set(glob.glob(os.path.join(download_dir, "*")))

def wait_new_file(download_dir, before, timeout=180):
    start = time.time()
    while time.time() - start < timeout:
        now = set(glob.glob(os.path.join(download_dir, "*")))
        new_files = now - before
        completed = [
            f for f in new_files
            if not f.endswith(".crdownload") and os.path.isfile(f)
        ]
        if completed:
            completed.sort(key=lambda x: os.path.getmtime(x), reverse=True)
            return completed[0]
        time.sleep(1)
    return None

def create_driver(download_dir, headless=True):
    chrome_options = Options()
    chrome_options.binary_location = "/usr/bin/google-chrome"

    prefs = {
        "download.default_directory": download_dir,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    }
    chrome_options.add_experimental_option("prefs", prefs)

    if headless:
        chrome_options.add_argument("--headless")

    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--remote-debugging-port=9222")
    chrome_options.add_argument("--window-size=1600,1200")
    chrome_options.add_argument("--lang=ko-KR")

    driver = webdriver.Chrome(options=chrome_options)

    try:
        driver.execute_cdp_cmd(
            "Page.setDownloadBehavior",
            {"behavior": "allow", "downloadPath": download_dir}
        )
    except Exception as e:
        print("download behavior setup skipped:", e)

    return driver

def open_page(driver):
    driver.get(URL)
    wait = WebDriverWait(driver, 30)
    wait.until(EC.presence_of_element_located((By.ID, "span_start_date_picker")))
    wait.until(EC.presence_of_element_located((By.ID, "span_end_date_picker")))
    wait.until(EC.presence_of_element_located((By.ID, "sido")))
    wait.until(EC.presence_of_element_located((By.ID, "sigun")))

def set_daily_mode(driver):
    driver.execute_script("document.getElementById('rdo3').checked = true;")
    driver.execute_script("document.getElementById('rdo4').checked = true;")
    driver.execute_script("document.getElementById('rdo3').dispatchEvent(new Event('change', {bubbles:true}));")
    driver.execute_script("document.getElementById('rdo4').dispatchEvent(new Event('change', {bubbles:true}));")

def set_region(driver, region_text):
    sido = Select(driver.find_element(By.ID, "sido"))

    matched = False
    for opt in sido.options:
        if opt.text.strip() == region_text:
            sido.select_by_visible_text(opt.text.strip())
            matched = True
            break

    if not matched:
        raise ValueError(f"시/도 옵션에 '{region_text}' 없음")

    WebDriverWait(driver, 20).until(
        lambda d: len(Select(d.find_element(By.ID, "sigun")).options) >= 1
    )
    time.sleep(2)

def get_sigun_list(driver):
    sigun = Select(driver.find_element(By.ID, "sigun"))
    values = []
    skip_texts = {"", "시/군/구", "선택", "전체", "선택하세요.", "선택하세요"}

    for opt in sigun.options:
        txt = opt.text.strip()
        val = (opt.get_attribute("value") or "").strip()

        if txt in skip_texts:
            continue
        if val == "":
            continue

        values.append(txt)

    return values

def set_sigun(driver, sigun_text):
    sel = Select(driver.find_element(By.ID, "sigun"))
    matched = False

    for opt in sel.options:
        txt = opt.text.strip()
        val = (opt.get_attribute("value") or "").strip()

        if txt == sigun_text and val != "":
            sel.select_by_visible_text(txt)
            matched = True
            break

    if not matched:
        raise ValueError(f"시/군/구 옵션에 '{sigun_text}' 없음")

    time.sleep(1)

def set_dates(driver, start_dt, end_dt):
    s = driver.find_element(By.ID, "span_start_date_picker")
    e = driver.find_element(By.ID, "span_end_date_picker")

    driver.execute_script("""
        arguments[0].removeAttribute('readonly');
        arguments[1].removeAttribute('readonly');
        arguments[0].value = arguments[2];
        arguments[1].value = arguments[3];
        arguments[0].dispatchEvent(new Event('input', {bubbles:true}));
        arguments[0].dispatchEvent(new Event('change', {bubbles:true}));
        arguments[1].dispatchEvent(new Event('input', {bubbles:true}));
        arguments[1].dispatchEvent(new Event('change', {bubbles:true}));
    """, s, e, start_dt, end_dt)

    time.sleep(1)

def click_csv_and_accept(driver):
    driver.execute_script("fn_Download(6);")
    WebDriverWait(driver, 10).until(EC.alert_is_present())
    alert = driver.switch_to.alert
    print("confirm:", alert.text)
    alert.accept()

def wait_processing_done(driver, timeout=180):
    end = time.time() + timeout
    while time.time() < end:
        try:
            modal = driver.find_element(By.ID, "modalwindow")
            style = modal.get_attribute("style") or ""
            visible = modal.is_displayed() and "display: none" not in style.lower()
            if visible:
                time.sleep(2)
                continue
        except:
            pass

        try:
            s = driver.find_element(By.ID, "span_start_date_picker")
            e = driver.find_element(By.ID, "span_end_date_picker")
            if s.is_enabled() and e.is_enabled():
                return
        except:
            pass

        time.sleep(2)

def rename_downloaded_file(path, download_dir, region, sigun, sdt, edt):
    ext = Path(path).suffix or ".csv"
    new_name = sanitize_filename(f"{region}_{sigun}_{sdt}_{edt}{ext}")
    new_path = os.path.join(download_dir, new_name)

    if os.path.exists(new_path):
        stem = Path(new_name).stem
        suf = Path(new_name).suffix
        i = 1
        while os.path.exists(os.path.join(download_dir, f"{stem}_{i}{suf}")):
            i += 1
        new_path = os.path.join(download_dir, f"{stem}_{i}{suf}")

    os.rename(path, new_path)
    return new_path

def collect_opinet_csv(region, start_year, end_year, region_price_root, headless=True):
    download_dir = os.path.join(region_price_root, region)
    os.makedirs(download_dir, exist_ok=True)

    print("DOWNLOAD_DIR:", download_dir)

    driver = create_driver(download_dir=download_dir, headless=headless)

    try:
        open_page(driver)
        set_daily_mode(driver)

        set_region(driver, region)
        sigun_list = get_sigun_list(driver)

        print(f"[REGION] {region}")
        print(f"[SIGUN COUNT] {len(sigun_list)}")
        print(sigun_list)

        ranges = make_ranges(start_year, end_year)
        print("[YEAR RANGES]")
        for y, sdt, edt in ranges:
            print(y, sdt, edt)

        for sigun_name in sigun_list:
            print(f"\n===== {region} / {sigun_name} =====")

            set_region(driver, region)
            set_sigun(driver, sigun_name)

            for y, sdt, edt in ranges:
                print(f" -> {y}: {sdt} ~ {edt}")

                set_dates(driver, sdt, edt)

                before = current_files(download_dir)

                try:
                    click_csv_and_accept(driver)
                except Exception as e:
                    print(f"    [ALERT/CLICK FAIL] {e}")
                    continue

                wait_processing_done(driver, timeout=180)
                new_file = wait_new_file(download_dir, before, timeout=180)

                if not new_file:
                    print("    [FAIL] 다운로드 파일 감지 실패")
                    continue

                saved = rename_downloaded_file(
                    path=new_file,
                    download_dir=download_dir,
                    region=region,
                    sigun=sigun_name,
                    sdt=sdt,
                    edt=edt
                )
                print("    [OK]", saved)
                time.sleep(1)

        print("\n=== 완료 ===")
        files = sorted(glob.glob(os.path.join(download_dir, "*")))
        print("총 파일 수:", len(files))
        for f in files[:20]:
            print("-", os.path.basename(f))

    finally:
        try:
            driver.quit()
        except:
            pass

### 2-3. 전국 지역별 다운로드 실행

In [ ]:
# =========================
# 3셀. 사용자 정의 + 실행
# =========================

REGION_LIST = ['울산', '서울', '경기', '강원', '충북', '충남', '전북', '전남', '경북', '경남', '부산', '제주', '대구', '인천', '광주', '대전', '세종']

# 이미 내려받은 지역이 있으면 여기에 입력해 재수집을 건너뜁니다. 예: ['서울', '경기']
DONE_LIST = []

for REGION in REGION_LIST:
    if REGION in DONE_LIST:
        print(f"[SKIP] {REGION}: DONE_LIST에 포함됨")
        continue

    START_YEAR = 2008
    END_YEAR = 2026
    HEADLESS = True

    collect_opinet_csv(
        region=REGION,
        start_year=START_YEAR,
        end_year=END_YEAR,
        region_price_root=REGION_PRICE_ROOT,
        headless=HEADLESS,
    )


### 2-4. 중복 다운로드 파일 점검

In [ ]:
import re
from pathlib import Path

# =========================
# 환경설정
# =========================
TARGET_BASE_DIR = Path(DATA_PATH) / "gas_station_prices_by_region"
DRY_RUN = True   # True면 삭제 안 하고 대상만 출력, False면 실제 삭제

# =========================
# 함수 정의
# =========================
def log(msg: str):
    print(msg)

def find_duplicate_files(base_dir: Path):
    """
    파일명 끝이 정확히 '_1.csv' 인 파일만 찾는다.
    예:
      서울_강남구_20080415_20081231_1.csv  -> 삭제 대상
      서울_강남구_20080415_20081231.csv    -> 유지
      test_10.csv                         -> 유지
    """
    pattern = re.compile(r"_1\.csv$", re.IGNORECASE)
    matched_files = []

    for file_path in base_dir.rglob("*.csv"):
        if pattern.search(file_path.name):
            matched_files.append(file_path)

    return sorted(matched_files)

def delete_files(file_list, dry_run=True):
    deleted = 0
    failed = 0

    for i, file_path in enumerate(file_list, 1):
        try:
            if dry_run:
                log(f"[DRY RUN] ({i}/{len(file_list)}) 삭제 대상: {file_path}")
            else:
                file_path.unlink()
                deleted += 1
                log(f"[DELETE] ({i}/{len(file_list)}) 삭제 완료: {file_path}")
        except Exception as e:
            failed += 1
            log(f"[ERROR] ({i}/{len(file_list)}) 삭제 실패: {file_path} | {e}")

    return deleted, failed

# =========================
# 사용자설정 & 실행
# =========================
if not TARGET_BASE_DIR.exists():
    raise FileNotFoundError(f"[ERROR] 대상 폴더가 존재하지 않음: {TARGET_BASE_DIR}")

log(f"[START] 대상 폴더: {TARGET_BASE_DIR}")
log("[STEP 1] '_1.csv' 파일 검색 시작")

target_files = find_duplicate_files(TARGET_BASE_DIR)

log(f"[INFO] 검색된 삭제 대상 파일 수: {len(target_files)}")

if len(target_files) == 0:
    log("[DONE] 삭제 대상 파일 없음")
else:
    log("[STEP 2] 삭제 실행")
    deleted_count, failed_count = delete_files(target_files, dry_run=DRY_RUN)

    if DRY_RUN:
        log("[DONE] DRY_RUN=True 상태라 실제 삭제는 하지 않음")
        log("[INFO] 실제 삭제하려면 DRY_RUN=False 로 바꿔서 다시 실행")
    else:
        log(f"[DONE] 삭제 완료 | 성공: {deleted_count} | 실패: {failed_count}")

### 2-5. 지역별 가격 시계열 및 메타데이터 생성

In [ ]:
# ============================================================
# 0. 환경설정
# ============================================================
import os
import re
import json
import unicodedata
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd


# ============================================================
# 1. 함수 정의
# ============================================================
def log(msg: str):
    print(msg)


def normalize_text(s: str) -> str:
    """
    한글 유니코드 정규화 + 공백 제거 + 소문자화
    """
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"\s+", "", s).strip().lower()
    return s


def find_region_dir(base_dir: Path, region_name: str) -> Path:
    """
    사용자 입력 region_name과 일치하는 지역 폴더 찾기
    - 유니코드 정규화 대응
    """
    if not base_dir.exists():
        raise FileNotFoundError(f"[ERROR] 지역 베이스 폴더가 존재하지 않음: {base_dir}")

    region_dirs = [p for p in base_dir.iterdir() if p.is_dir()]
    if not region_dirs:
        raise FileNotFoundError(f"[ERROR] 지역 폴더가 없음: {base_dir}")

    target = normalize_text(region_name)

    # 1차: 정규화 후 완전일치
    for p in region_dirs:
        if normalize_text(p.name) == target:
            return p

    # 2차: 부분일치
    candidates = []
    for p in region_dirs:
        pn = normalize_text(p.name)
        if target in pn or pn in target:
            candidates.append(p)

    if len(candidates) == 1:
        return candidates[0]

    available = [unicodedata.normalize("NFC", p.name) for p in region_dirs]
    raise FileNotFoundError(
        f"[ERROR] 지역 폴더를 찾지 못함: {region_name}\n"
        f"[INFO] 사용 가능한 지역 폴더: {available}"
    )


def list_csv_files(region_dir: Path) -> List[Path]:
    """
    지역 폴더 내 모든 csv 파일 수집
    """
    files = sorted(region_dir.rglob("*.csv"))
    if not files:
        raise FileNotFoundError(f"[ERROR] CSV 파일이 없음: {region_dir}")
    return files


def read_csv_auto(file_path: Path) -> pd.DataFrame:
    """
    CSV 인코딩 자동 시도
    """
    encodings = ["cp949", "utf-8-sig", "euc-kr", "utf-8"]
    last_error = None

    for enc in encodings:
        try:
            df = pd.read_csv(file_path, encoding=enc)
            return df
        except Exception as e:
            last_error = e

    raise RuntimeError(f"[ERROR] 파일 읽기 실패: {file_path}\n원인: {last_error}")


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    컬럼명 공백 제거
    """
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def validate_required_columns(df: pd.DataFrame, file_path: Path):
    required_cols = ["번호", "지역", "상호", "주소", "기간", "상표", "셀프여부", "휘발유", "경유"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"[ERROR] 필수 컬럼 누락: {missing} | 파일: {file_path}")


def clean_raw_station_price_df(df: pd.DataFrame, file_path: Path) -> pd.DataFrame:
    """
    원본 파일 정리:
    - 메타 행 제거
    - 필요한 컬럼만 선택
    - 날짜/가격 형 변환
    - 표준 컬럼명으로 변경
    """
    df = standardize_columns(df)
    validate_required_columns(df, file_path)

    use_cols = ["번호", "지역", "상호", "주소", "기간", "상표", "셀프여부", "휘발유", "경유"]
    df = df[use_cols].copy()

    # 메타 행 제거: 번호가 A + 숫자 형태인 행만 유지
    before_rows = len(df)
    df["번호"] = df["번호"].astype(str).str.strip()
    df = df[df["번호"].str.match(r"^A\d+$", na=False)].copy()
    after_rows = len(df)

    # 문자열 컬럼 정리
    text_cols = ["지역", "상호", "주소", "상표", "셀프여부"]
    for c in text_cols:
        df[c] = df[c].astype(str).str.strip()
        df[c] = df[c].replace({"nan": np.nan, "None": np.nan, "": np.nan})

    # 날짜 정리: 20080425.0 -> 20080425 -> datetime
    period_num = pd.to_numeric(df["기간"], errors="coerce")
    period_str = period_num.astype("Int64").astype(str).replace("<NA>", np.nan)

    df["date"] = pd.to_datetime(
        period_str,
        format="%Y%m%d",
        errors="coerce"
    )

    # 가격 숫자형
    df["휘발유"] = pd.to_numeric(df["휘발유"], errors="coerce")
    df["경유"] = pd.to_numeric(df["경유"], errors="coerce")

    # 표준 컬럼명
    df = df.rename(columns={
        "번호": "station_id",
        "지역": "region",
        "상호": "station_name",
        "주소": "address",
        "상표": "brand",
        "셀프여부": "is_self_raw",
        "휘발유": "price_gasoline",
        "경유": "price_diesel",
    })

    # 유효 날짜만 유지
    invalid_date_cnt = int(df["date"].isna().sum())
    df = df[df["date"].notna()].copy()

    # 날짜 문자열
    df["date_str"] = df["date"].dt.strftime("%Y-%m-%d")

    # 원본 파일명
    df["source_file"] = file_path.name

    log(
        f"[CLEAN] {file_path.name} | 원본 {before_rows}행 -> 메타제거 후 {after_rows}행 "
        f"| 날짜오류 제거 {invalid_date_cnt}행 | 최종 {len(df)}행"
    )
    return df


def build_price_matrix(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    """
    index: yyyy-mm-dd
    columns: station_id
    values: fuel price
    - 동일 station_id/date 중복 시 마지막 값 사용
    - 전체 날짜 구간으로 reindex
    """
    temp = df[["date", "station_id", value_col]].copy()
    temp = temp.sort_values(["date", "station_id"]).drop_duplicates(
        subset=["date", "station_id"], keep="last"
    )

    wide = temp.pivot(index="date", columns="station_id", values=value_col)
    wide = wide.sort_index()
    wide = wide.sort_index(axis=1)

    if len(wide.index) > 0:
        full_idx = pd.date_range(wide.index.min(), wide.index.max(), freq="D")
        wide = wide.reindex(full_idx)

    wide.index.name = "date"
    wide.index = pd.Index(wide.index.strftime("%Y-%m-%d"), name="date")
    return wide


def extract_change_history_fast(group_df: pd.DataFrame, col: str) -> List[List[object]]:
    """
    group_df는 이미 date 기준 정렬되어 있다고 가정
    반환:
      [["2008-04-25", "일반"], ["2015-07-12", "셀프"], ...]
    """
    s = group_df[col]

    # 결측 제거
    mask = s.notna()
    if not mask.any():
        return []

    dates = group_df.loc[mask, "date_str"].to_numpy()
    values = s.loc[mask].astype(str).str.strip().to_numpy()

    # 빈 문자열 제거
    valid_mask = values != ""
    if not valid_mask.any():
        return []

    dates = dates[valid_mask]
    values = values[valid_mask]

    if len(values) == 0:
        return []

    # 연속 중복 제거
    change_mask = np.empty(len(values), dtype=bool)
    change_mask[0] = True
    if len(values) > 1:
        change_mask[1:] = values[1:] != values[:-1]

    changed_dates = dates[change_mask]
    changed_values = values[change_mask]

    return [[d, v] for d, v in zip(changed_dates.tolist(), changed_values.tolist())]


def build_metadata_json(df: pd.DataFrame) -> Dict[str, Dict[str, List[List[object]]]]:
    """
    빠른 메타 JSON 생성
    구조:
    {
      "A0006683": {
        "region": [["2008-04-25", "경기 가평군"]],
        "station_name": [["2008-04-25", "(주)설악주유소"]],
        "address": [["2008-04-25", "경기 가평군 설악면 유명로 1711"]],
        "brand": [["2008-04-25", "HD현대오일뱅크"]],
        "is_self": [["2008-04-25", "일반"], ["2015-07-12", "셀프"]]
      }
    }
    """
    use_cols = [
        "station_id", "date", "date_str",
        "region", "station_name", "address", "brand", "is_self_raw"
    ]

    work = df[use_cols].copy()
    work = work.sort_values(["station_id", "date", "date_str"])
    work = work.drop_duplicates(subset=["station_id", "date"], keep="last")

    meta = {}
    grouped = work.groupby("station_id", sort=True)
    total = work["station_id"].nunique()

    for idx, (station_id, g) in enumerate(grouped, 1):
        if idx % 200 == 0 or idx == 1 or idx == total:
            log(f"[META] 메타 정보 생성 중: {idx}/{total} | station_id={station_id}")

        meta[station_id] = {
            "region": extract_change_history_fast(g, "region"),
            "station_name": extract_change_history_fast(g, "station_name"),
            "address": extract_change_history_fast(g, "address"),
            "brand": extract_change_history_fast(g, "brand"),
            "is_self": extract_change_history_fast(g, "is_self_raw"),
        }

    return meta


def save_dataframe_csv(df: pd.DataFrame, save_path: Path):
    save_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(save_path, encoding="utf-8-sig")


def save_json(data: dict, save_path: Path, pretty: bool = True):
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        if pretty:
            json.dump(data, f, ensure_ascii=False, indent=2)
        else:
            json.dump(data, f, ensure_ascii=False)


def run_region_integration(
    base_input_dir: Path,
    base_output_dir: Path,
    region_name: str,
    gasoline_filename: str,
    diesel_filename: str,
    metadata_filename: str,
    pretty_json: bool = True,
):
    log("=" * 80)
    log("[START] 지역별 주유소 가격 통합 시작")
    log(f"[INFO] 입력 베이스 폴더: {base_input_dir}")
    log(f"[INFO] 출력 베이스 폴더: {base_output_dir}")
    log(f"[INFO] 대상 지역 입력값: {region_name}")

    # 지역 폴더 찾기
    region_dir = find_region_dir(base_input_dir, region_name)
    log(f"[INFO] 매칭된 지역 폴더: {region_dir}")

    # 파일 수집
    csv_files = list_csv_files(region_dir)
    log(f"[INFO] CSV 파일 수: {len(csv_files)}")

    # 파일별 처리
    all_list = []
    fail_logs = []

    for i, file_path in enumerate(csv_files, 1):
        try:
            log(f"[READ] ({i}/{len(csv_files)}) {file_path.name}")
            raw = read_csv_auto(file_path)
            cleaned = clean_raw_station_price_df(raw, file_path)
            all_list.append(cleaned)
        except Exception as e:
            fail_logs.append({
                "file": file_path.name,
                "error": str(e)
            })
            log(f"[ERROR] 파일 처리 실패: {file_path.name} | {e}")

    if not all_list:
        raise RuntimeError("[ERROR] 처리 가능한 파일이 하나도 없음")

    # 전체 병합
    log("[MERGE] 전체 파일 병합 중")
    df_all = pd.concat(all_list, ignore_index=True)
    log(f"[INFO] 병합 후 행 수: {len(df_all):,}")

    # station_id + date 기준 마지막 값 유지
    before_dedup = len(df_all)
    df_all = df_all.sort_values(["date", "station_id", "source_file"]).drop_duplicates(
        subset=["station_id", "date"], keep="last"
    ).copy()
    after_dedup = len(df_all)
    log(f"[INFO] 중복 제거: {before_dedup:,} -> {after_dedup:,}")

    # 가격 시계열 생성
    log("[PIVOT] 휘발유 시계열 생성 중")
    gasoline_wide = build_price_matrix(df_all, "price_gasoline")
    log(f"[INFO] 휘발유 shape: {gasoline_wide.shape}")

    log("[PIVOT] 경유 시계열 생성 중")
    diesel_wide = build_price_matrix(df_all, "price_diesel")
    log(f"[INFO] 경유 shape: {diesel_wide.shape}")

    # 메타 JSON 생성
    log("[META] 메타 JSON 생성 중")
    metadata = build_metadata_json(df_all)

    # 저장 경로
    out_region_dir = base_output_dir / unicodedata.normalize("NFC", region_dir.name)
    gasoline_path = out_region_dir / gasoline_filename
    diesel_path = out_region_dir / diesel_filename
    metadata_path = out_region_dir / metadata_filename
    fail_log_path = out_region_dir / "failed_files_log.csv"

    # 저장
    log("[SAVE] 휘발유 CSV 저장 중")
    save_dataframe_csv(gasoline_wide, gasoline_path)

    log("[SAVE] 경유 CSV 저장 중")
    save_dataframe_csv(diesel_wide, diesel_path)

    log("[SAVE] 메타 JSON 저장 중")
    save_json(metadata, metadata_path, pretty=pretty_json)

    if fail_logs:
        log("[SAVE] 실패 로그 저장 중")
        pd.DataFrame(fail_logs).to_csv(fail_log_path, index=False, encoding="utf-8-sig")

    # 요약 출력
    log("=" * 80)
    log("[DONE] 지역별 통합 완료")
    log(f"[SAVE] 휘발유 CSV : {gasoline_path}")
    log(f"[SAVE] 경유 CSV   : {diesel_path}")
    log(f"[SAVE] 메타 JSON  : {metadata_path}")
    if fail_logs:
        log(f"[SAVE] 실패 로그   : {fail_log_path}")

    log(f"[SUMMARY] 날짜 범위: {df_all['date'].min().strftime('%Y-%m-%d')} ~ {df_all['date'].max().strftime('%Y-%m-%d')}")
    log(f"[SUMMARY] 주유소 수: {df_all['station_id'].nunique():,}")
    log(f"[SUMMARY] 총 레코드 수: {len(df_all):,}")
    log(f"[SUMMARY] 실패 파일 수: {len(fail_logs)}")

    print("\n[PREVIEW] 휘발유 상위 5행")
    display(gasoline_wide.head())

    print("\n[PREVIEW] 경유 상위 5행")
    display(diesel_wide.head())

    sample_keys = list(metadata.keys())[:3]
    sample_meta = {k: metadata[k] for k in sample_keys}
    print("\n[PREVIEW] 메타 JSON 샘플 3개")
    print(json.dumps(sample_meta, ensure_ascii=False, indent=2))


# ============================================================
# 2. 사용자설정 & 실행
# ============================================================
# 입력 폴더
INPUT_BASE_DIR = Path(DATA_PATH) / "gas_station_prices_by_region"

# 출력 폴더
OUTPUT_BASE_DIR = Path(PROCESSED_PATH) / "additional_data" / "gas_station_prices_by_region"

# 처리할 지역명
TARGET_REGIONS = ['강원', '경기', '경남', '경북', '광주', '대구', '대전', '부산', '서울', '세종', '울산', '인천', '전남', '전북', '제주', '충남', '충북']

# 저장 파일명
GASOLINE_FILENAME = "gasoline.csv"
DIESEL_FILENAME = "diesel.csv"
METADATA_FILENAME = "metadata.json"

# True면 사람이 보기 좋게 들여쓰기 저장, False면 더 빠르게 저장
PRETTY_JSON = True

for TARGET_REGION in TARGET_REGIONS:
    print("=" * 30 + f" {TARGET_REGION} " + "=" * 30)
    run_region_integration(
        base_input_dir=INPUT_BASE_DIR,
        base_output_dir=OUTPUT_BASE_DIR,
        region_name=TARGET_REGION,
        gasoline_filename=GASOLINE_FILENAME,
        diesel_filename=DIESEL_FILENAME,
        metadata_filename=METADATA_FILENAME,
        pretty_json=PRETTY_JSON,
    )


### 2-6. 주유소 메타데이터 좌표 보강

In [ ]:
# =========================
# 0. 환경설정
# =========================
import os
import re
import json
import time
import math
from copy import deepcopy
from difflib import SequenceMatcher

import requests

# PROCESSED_PATH 는 이미 상위 셀에서 선언되어 있다고 가정

# =========================
# 1. 함수 정의
# =========================
def log(msg: str):
    print(msg, flush=True)

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(obj, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def normalize_space(text: str) -> str:
    if text is None:
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def strip_parentheses(text: str) -> str:
    text = normalize_space(text)
    text = re.sub(r"\([^)]*\)", "", text)
    text = normalize_space(text)
    return text

def normalize_compare_text(text: str) -> str:
    text = normalize_space(text)
    text = strip_parentheses(text)
    text = text.replace("특별시", "").replace("광역시", "").replace("특별자치시", "").replace("특별자치도", "")
    text = text.replace("경기도", "경기").replace("강원특별자치도", "강원").replace("제주특별자치도", "제주")
    text = text.replace(",", " ")
    text = re.sub(r"[^0-9a-zA-Z가-힣\- ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def similarity(a: str, b: str) -> float:
    a = normalize_compare_text(a)
    b = normalize_compare_text(b)
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()

def build_query_candidates(address: str):
    cands = []
    a0 = normalize_space(address)
    a1 = strip_parentheses(a0)
    a2 = normalize_compare_text(a0)

    for a in [a0, a1, a2]:
        a = normalize_space(a)
        if a and a not in cands:
            cands.append(a)

    return cands[:3]

def safe_request_json(url: str, params: dict, timeout: int = 30):
    res = requests.get(url, params=params, timeout=timeout)
    res.raise_for_status()
    try:
        return res.json()
    except Exception:
        return json.loads(res.text)

def call_addr_search_api(address: str, api_key: str, timeout: int = 30):
    url = "https://business.juso.go.kr/addrlink/addrLinkApi.do"
    params = {
        "confmKey": api_key,
        "currentPage": "1",
        "countPerPage": "10",
        "keyword": address,
        "resultType": "json",
        "hstryYn": "Y",
        "firstSort": "road",
    }
    data = safe_request_json(url, params, timeout=timeout)
    results = data.get("results", {})
    common = results.get("common", {})
    juso_list = results.get("juso", [])
    error_code = str(common.get("errorCode", ""))
    error_message = str(common.get("errorMessage", ""))

    if error_code not in ("0", ""):
        return {"ok": False, "message": f"{error_code} / {error_message}", "items": []}

    return {"ok": True, "message": "", "items": juso_list}

def select_best_search_result(query_address: str, juso_list: list):
    if not juso_list:
        return None

    best_item = None
    best_score = -1.0
    q = normalize_compare_text(query_address)
    q_nums = re.findall(r"\d+(?:-\d+)?", q)

    for item in juso_list:
        road = item.get("roadAddr", "") or ""
        jibun = item.get("jibunAddr", "") or ""
        road_p1 = item.get("roadAddrPart1", "") or ""
        bd_nm = item.get("bdNm", "") or ""

        score = max(
            similarity(q, road),
            similarity(q, jibun),
            similarity(q, road_p1),
        )

        candidate_text = normalize_compare_text(" ".join([road, jibun, road_p1, bd_nm]))

        if q == normalize_compare_text(road) or q == normalize_compare_text(jibun):
            score += 1.0

        if q_nums:
            hit = sum(1 for num in q_nums if num in candidate_text)
            score += 0.05 * hit

        if score > best_score:
            best_score = score
            best_item = item

    return best_item

def search_best_road_address_with_juso(address: str, api_key: str, sleep_sec: float = 0.03):
    query_candidates = build_query_candidates(address)

    best_selected = None
    best_selected_score = -1.0
    best_selected_query = None

    for q in query_candidates:
        time.sleep(sleep_sec)
        search_res = call_addr_search_api(q, api_key)

        if not search_res["ok"]:
            continue

        items = search_res["items"]
        if not items:
            continue

        candidate_best = select_best_search_result(address, items)
        if candidate_best is None:
            continue

        road = candidate_best.get("roadAddr", "") or ""
        jibun = candidate_best.get("jibunAddr", "") or ""
        score = max(similarity(address, road), similarity(address, jibun))

        if score > best_selected_score:
            best_selected_score = score
            best_selected = candidate_best
            best_selected_query = q

    if best_selected is None:
        return {
            "ok": False,
            "roadAddr": "",
            "jibunAddr": "",
            "selected_query": None,
            "message": "JUSO 검색 결과 없음",
        }

    return {
        "ok": True,
        "roadAddr": best_selected.get("roadAddr", "") or "",
        "jibunAddr": best_selected.get("jibunAddr", "") or "",
        "selected_query": best_selected_query,
        "message": "",
    }

def call_geocoder_batch(addresses: list, token: str, timeout: int = 60, max_retry: int = 3, retry_sleep_sec: float = 1.0):
    url = "https://geocode-api.gimi9.com/geocode"
    headers = {
        "Content-Type": "application/json",
        "Authorization": token,
    }
    payload = {"q": addresses}
    last_error = None

    for attempt in range(1, max_retry + 1):
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=timeout)

            if res.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"retryable_status={res.status_code}, body={res.text[:500]}")

            res.raise_for_status()
            return res.json()

        except Exception as e:
            last_error = e
            if attempt == max_retry:
                break
            time.sleep(retry_sleep_sec * attempt)

    raise last_error

def normalize_geocoder_item(item: dict, original_input: str, source_label: str):
    if item is None:
        return {
            "ok": False,
            "latitude": None,
            "longitude": None,
            "message": "지오코더 응답 누락",
            "errmsg": "지오코더 응답 누락",
            "bld_mgt_no": "",
            "addressCls": "",
            "hd_nm": "",
            "source_label": source_label,
            "source_input": original_input,
            "raw": None,
        }

    ok = bool(item.get("success", False))
    latitude = item.get("y_axis", None)
    longitude = item.get("x_axis", None)
    errmsg = normalize_space(item.get("errmsg", ""))

    if ok:
        message = ""
    else:
        message = errmsg if errmsg else "지오코딩 실패"

    return {
        "ok": ok,
        "latitude": float(latitude) if latitude is not None else None,
        "longitude": float(longitude) if longitude is not None else None,
        "message": message,
        "errmsg": errmsg,
        "bld_mgt_no": normalize_space(item.get("bld_mgt_no", "")),
        "addressCls": normalize_space(item.get("addressCls", "")),
        "hd_nm": normalize_space(item.get("hd_nm", "")),
        "source_label": source_label,
        "source_input": original_input,
        "raw": item,
    }

def geocode_addresses_with_geocoder(addresses: list, token: str, batch_size: int = 1000, timeout: int = 60, max_retry: int = 3, retry_sleep_sec: float = 1.0, progress_name: str = "GEOCODER"):
    cache = {}
    total = len(addresses)

    if total == 0:
        return cache

    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        batch = addresses[start:end]
        batch_no = start // batch_size + 1
        batch_total = math.ceil(total / batch_size)

        log(f"[{progress_name}] batch {batch_no}/{batch_total} | rows={len(batch):,} | progress={end:,}/{total:,}")

        try:
            data = call_geocoder_batch(
                addresses=batch,
                token=token,
                timeout=timeout,
                max_retry=max_retry,
                retry_sleep_sec=retry_sleep_sec,
            )
            results = data.get("results", [])
        except Exception as e:
            for addr in batch:
                cache[addr] = {
                    "ok": False,
                    "latitude": None,
                    "longitude": None,
                    "message": f"지오코더 요청 예외: {repr(e)}",
                    "errmsg": f"지오코더 요청 예외: {repr(e)}",
                    "bld_mgt_no": "",
                    "addressCls": "",
                    "hd_nm": "",
                    "source_label": progress_name,
                    "source_input": addr,
                    "raw": None,
                }
            continue

        if len(results) == len(batch):
            for addr, item in zip(batch, results):
                cache[addr] = normalize_geocoder_item(item, addr, progress_name)
        else:
            results_by_input = {normalize_space(item.get("inputaddr", "")): item for item in results}
            for addr in batch:
                cache[addr] = normalize_geocoder_item(results_by_input.get(normalize_space(addr)), addr, progress_name)

    return cache

def geocoder_confidence_score(result: dict) -> int:
    if result is None or not result.get("ok", False):
        return 0

    score = 0
    score += 100
    if result.get("bld_mgt_no"):
        score += 20
    if result.get("addressCls") == "ROAD_ADDRESS":
        score += 5
    if not result.get("errmsg"):
        score += 5
    if result.get("hd_nm"):
        score += 1
    return score

def needs_secondary_check(primary_result: dict) -> bool:
    if primary_result is None:
        return True
    if not primary_result.get("ok", False):
        return True
    if not primary_result.get("bld_mgt_no"):
        return True
    if primary_result.get("errmsg"):
        return True
    return False

def haversine_m(lat1, lon1, lat2, lon2):
    if None in (lat1, lon1, lat2, lon2):
        return None

    r = 6371000.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)

    a = math.sin(dlat / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlon / 2) ** 2
    c = 2 * math.asin(math.sqrt(a))
    return r * c

def choose_final_result(original_address: str, primary_result: dict, secondary_result: dict, juso_meta: dict, distance_tolerance_m: float = 30.0):
    decision = {
        "decision": "",
        "distance_m": None,
        "primary_score": geocoder_confidence_score(primary_result),
        "secondary_score": geocoder_confidence_score(secondary_result),
        "review_needed": False,
    }

    if primary_result is None:
        primary_result = {"ok": False, "message": "primary 없음", "latitude": None, "longitude": None}
    if secondary_result is None:
        secondary_result = {"ok": False, "message": "secondary 없음", "latitude": None, "longitude": None}

    if primary_result.get("ok") and not secondary_result.get("ok"):
        decision["decision"] = "primary_only"
        return primary_result, decision

    if (not primary_result.get("ok")) and secondary_result.get("ok"):
        decision["decision"] = "secondary_fallback"
        return secondary_result, decision

    if (not primary_result.get("ok")) and (not secondary_result.get("ok")):
        decision["decision"] = "both_failed"
        return primary_result, decision

    distance_m = haversine_m(
        primary_result.get("latitude"),
        primary_result.get("longitude"),
        secondary_result.get("latitude"),
        secondary_result.get("longitude"),
    )
    decision["distance_m"] = distance_m

    if distance_m is not None and distance_m <= distance_tolerance_m:
        decision["decision"] = "same_within_tolerance"
        return primary_result, decision

    decision["review_needed"] = True

    primary_score = decision["primary_score"]
    secondary_score = decision["secondary_score"]

    if secondary_score > primary_score:
        decision["decision"] = "secondary_better_confidence"
        return secondary_result, decision

    if primary_score > secondary_score:
        decision["decision"] = "primary_better_confidence"
        return primary_result, decision

    standardized_road = normalize_compare_text((juso_meta or {}).get("roadAddr", ""))
    original_norm = normalize_compare_text(original_address)

    if standardized_road and standardized_road != original_norm:
        decision["decision"] = "secondary_standardized_tiebreak"
        return secondary_result, decision

    decision["decision"] = "primary_tiebreak"
    return primary_result, decision

def print_records(title: str, records: list, print_all: bool = False, limit: int = 20):
    log(f"[{title}] {len(records):,}건")
    if len(records) == 0:
        return

    show_records = records if print_all else records[:limit]
    for rec in show_records:
        print(json.dumps(rec, ensure_ascii=False, indent=2), flush=True)

    if (not print_all) and len(records) > limit:
        log(f"[{title}] 나머지 {len(records) - limit:,}건은 변수에만 저장됨")

# =========================
# 2. 사용자정의 & 실행
# =========================
REGION = ['강원','경기','경남','경북','광주','대구','대전','부산','서울','세종','울산','인천','전남','전북','제주','충남','충북']
for REGION_NAME in REGION:
    print('=' * 20 + f'{REGION_NAME}'+ '=' * 20 )
    BASE_DIR = os.path.join(PROCESSED_PATH, "additional_data", "gas_station_prices_by_region", REGION_NAME)
    INPUT_PATH = os.path.join(BASE_DIR, "metadata.json")
    OUTPUT_PATH = os.path.join(BASE_DIR, "metadata__latlon.json")

    # 1) 지오코더 메인 토큰
    GEOCODER_TOKEN = ""  # 필수: 사용하는 지오코더 서비스 토큰을 입력하세요.

    # 2) JUSO 도로명주소 검색 API 키 (secondary 사용 시에만 필요)
    JUSO_SEARCH_API_KEY = ""  # 선택: SECONDARY_MODE가 off가 아닐 때 도로명주소 검색 API 키를 입력하세요.


    if not GEOCODER_TOKEN:
        raise ValueError("GEOCODER_TOKEN을 입력해야 주유소 메타데이터 좌표 보강을 실행할 수 있습니다.")

    # secondary 모드
    # - "off"           : 지오코더 단독만 사용
    # - "fallback_only" : 지오코더 실패/저신뢰 건만 JUSO 검색 -> 지오코더 재시도
    # - "compare_all"   : 모든 주소에 대해 지오코더 단독 + JUSO검색후지오코더 비교
    SECONDARY_MODE = "off"

    # 지오코더 배치 설정
    GEOCODER_BATCH_SIZE = 1000
    GEOCODER_TIMEOUT = 60
    GEOCODER_MAX_RETRY = 3
    GEOCODER_RETRY_SLEEP_SEC = 1.0

    # JUSO 검색 호출 간격
    JUSO_SEARCH_SLEEP_SEC = 0.03

    # 비교 허용 오차 (미터)
    DISTANCE_TOLERANCE_M = 30.0

    # 실패/충돌 출력 설정
    PRINT_ALL_FAILED = False
    PRINT_ALL_CONFLICTS = False
    PRINT_LIMIT = 30

    # 메인 결과 JSON 저장 여부
    SAVE_OUTPUT_JSON = True

    log("[1/7] 입력 파일 확인")
    if not os.path.exists(INPUT_PATH):
        raise FileNotFoundError(f"입력 파일이 없습니다: {INPUT_PATH}")
    ensure_dir(BASE_DIR)

    log("[2/7] metadata.json 로드")
    metadata = load_json(INPUT_PATH)
    log(f"  - region: {REGION_NAME}")
    log(f"  - station 수: {len(metadata):,}")

    log("[3/7] 주소 이력 수집")
    all_address_items = []
    for station_id, info in metadata.items():
        address_history = info.get("address", [])
        if isinstance(address_history, list):
            for item in address_history:
                if isinstance(item, list) and len(item) >= 2:
                    start_date = str(item[0]).strip()
                    addr = normalize_space(item[1])
                    if addr:
                        all_address_items.append((station_id, start_date, addr))

    unique_addresses = sorted({normalize_space(addr) for _, _, addr in all_address_items if normalize_space(addr)})
    log(f"  - 전체 주소 이력 수: {len(all_address_items):,}")
    log(f"  - 고유 주소 수: {len(unique_addresses):,}")

    log("[4/7] 1차 지오코더(메인) 실행")
    primary_cache = geocode_addresses_with_geocoder(
        addresses=unique_addresses,
        token=GEOCODER_TOKEN,
        batch_size=GEOCODER_BATCH_SIZE,
        timeout=GEOCODER_TIMEOUT,
        max_retry=GEOCODER_MAX_RETRY,
        retry_sleep_sec=GEOCODER_RETRY_SLEEP_SEC,
        progress_name="GEOCODER_PRIMARY",
    )

    primary_success_cnt = sum(1 for v in primary_cache.values() if v.get("ok"))
    primary_fail_cnt = len(primary_cache) - primary_success_cnt
    log(f"  - 1차 성공: {primary_success_cnt:,}")
    log(f"  - 1차 실패: {primary_fail_cnt:,}")

    secondary_input_addresses = []
    if SECONDARY_MODE == "compare_all":
        secondary_input_addresses = unique_addresses[:]
    elif SECONDARY_MODE == "fallback_only":
        secondary_input_addresses = [addr for addr in unique_addresses if needs_secondary_check(primary_cache.get(addr))]
    else:
        secondary_input_addresses = []

    log("[5/7] secondary 대상 선정")
    log(f"  - SECONDARY_MODE = {SECONDARY_MODE}")
    log(f"  - secondary 대상 수 = {len(secondary_input_addresses):,}")

    juso_cache = {}
    secondary_cache = {}

    if secondary_input_addresses:
        if not JUSO_SEARCH_API_KEY:
            raise ValueError("SECONDARY_MODE가 off가 아닌데 JUSO_SEARCH_API_KEY가 비어 있습니다.")

        log("[5-1/7] JUSO 검색으로 도로명주소 표준화")
        for idx, addr in enumerate(secondary_input_addresses, start=1):
            if idx == 1 or idx % 200 == 0 or idx == len(secondary_input_addresses):
                log(f"  - JUSO 진행률: {idx:,}/{len(secondary_input_addresses):,}")

            try:
                juso_cache[addr] = search_best_road_address_with_juso(
                    address=addr,
                    api_key=JUSO_SEARCH_API_KEY,
                    sleep_sec=JUSO_SEARCH_SLEEP_SEC,
                )
            except Exception as e:
                juso_cache[addr] = {
                    "ok": False,
                    "roadAddr": "",
                    "jibunAddr": "",
                    "selected_query": None,
                    "message": f"JUSO 검색 예외: {repr(e)}",
                }

        secondary_road_addresses = sorted({
            v["roadAddr"]
            for v in juso_cache.values()
            if v.get("ok") and normalize_space(v.get("roadAddr", ""))
        })

        log("[5-2/7] 표준화된 roadAddr를 지오코더로 재지오코딩")
        log(f"  - 고유 roadAddr 수 = {len(secondary_road_addresses):,}")

        secondary_geo_by_road = geocode_addresses_with_geocoder(
            addresses=secondary_road_addresses,
            token=GEOCODER_TOKEN,
            batch_size=GEOCODER_BATCH_SIZE,
            timeout=GEOCODER_TIMEOUT,
            max_retry=GEOCODER_MAX_RETRY,
            retry_sleep_sec=GEOCODER_RETRY_SLEEP_SEC,
            progress_name="GEOCODER_SECONDARY",
        )

        for original_addr in secondary_input_addresses:
            juso_meta = juso_cache.get(original_addr, {})
            if juso_meta.get("ok") and juso_meta.get("roadAddr"):
                road_addr = juso_meta["roadAddr"]
                secondary_result = deepcopy(secondary_geo_by_road.get(road_addr, {
                    "ok": False,
                    "latitude": None,
                    "longitude": None,
                    "message": "secondary geocoder 응답 누락",
                    "errmsg": "secondary geocoder 응답 누락",
                    "bld_mgt_no": "",
                    "addressCls": "",
                    "hd_nm": "",
                    "source_label": "GEOCODER_SECONDARY",
                    "source_input": road_addr,
                    "raw": None,
                }))
                secondary_result["standardized_roadAddr"] = road_addr
                secondary_result["juso_jibunAddr"] = juso_meta.get("jibunAddr", "")
                secondary_result["juso_selected_query"] = juso_meta.get("selected_query")
                secondary_cache[original_addr] = secondary_result
            else:
                secondary_cache[original_addr] = {
                    "ok": False,
                    "latitude": None,
                    "longitude": None,
                    "message": juso_meta.get("message", "JUSO 표준화 실패"),
                    "errmsg": juso_meta.get("message", "JUSO 표준화 실패"),
                    "bld_mgt_no": "",
                    "addressCls": "",
                    "hd_nm": "",
                    "source_label": "GEOCODER_SECONDARY",
                    "source_input": original_addr,
                    "raw": None,
                    "standardized_roadAddr": "",
                    "juso_jibunAddr": "",
                    "juso_selected_query": None,
                }

    log("[6/7] 최종 좌표 선택")
    final_cache = {}
    conflict_records = []

    for addr in unique_addresses:
        primary_result = primary_cache.get(addr)
        secondary_result = secondary_cache.get(addr) if addr in secondary_cache else None
        juso_meta = juso_cache.get(addr, {})

        chosen, decision = choose_final_result(
            original_address=addr,
            primary_result=primary_result,
            secondary_result=secondary_result,
            juso_meta=juso_meta,
            distance_tolerance_m=DISTANCE_TOLERANCE_M,
        )

        final_entry = deepcopy(chosen)
        final_entry["decision"] = decision["decision"]
        final_entry["distance_m"] = decision["distance_m"]
        final_entry["review_needed"] = decision["review_needed"]

        if final_entry.get("source_label") == "GEOCODER_PRIMARY":
            final_entry["selected_source"] = "geocoder_only"
        elif final_entry.get("source_label") == "GEOCODER_SECONDARY":
            final_entry["selected_source"] = "juso_search_then_geocoder"
        else:
            final_entry["selected_source"] = "unknown"

        if decision["review_needed"]:
            conflict_records.append({
                "address": addr,
                "decision": decision["decision"],
                "distance_m": decision["distance_m"],
                "primary": {
                    "ok": None if primary_result is None else primary_result.get("ok"),
                    "lat": None if primary_result is None else primary_result.get("latitude"),
                    "lon": None if primary_result is None else primary_result.get("longitude"),
                    "bld_mgt_no": None if primary_result is None else primary_result.get("bld_mgt_no"),
                    "errmsg": None if primary_result is None else primary_result.get("errmsg"),
                },
                "secondary": {
                    "ok": None if secondary_result is None else secondary_result.get("ok"),
                    "lat": None if secondary_result is None else secondary_result.get("latitude"),
                    "lon": None if secondary_result is None else secondary_result.get("longitude"),
                    "bld_mgt_no": None if secondary_result is None else secondary_result.get("bld_mgt_no"),
                    "errmsg": None if secondary_result is None else secondary_result.get("errmsg"),
                    "standardized_roadAddr": None if secondary_result is None else secondary_result.get("standardized_roadAddr"),
                },
            })

        final_cache[addr] = final_entry

    log("[7/7] location 생성 및 저장")
    output_metadata = deepcopy(metadata)
    failed_records = []

    for station_id, info in output_metadata.items():
        original_info = deepcopy(metadata.get(station_id, {}))   # 원본 metadata 보존
        location_history = []
        address_history = info.get("address", [])

        if not isinstance(address_history, list) or len(address_history) == 0:
            info["location"] = []
            failed_records.append({
                "station_id": station_id,
                "start_date": None,
                "address": None,
                "message": "address 이력이 비어 있음",
                "selected_source": None,

                # 원본 metadata 주요 정보
                "station_name": original_info.get("station_name"),
                "region": original_info.get("region"),
                "brand": original_info.get("brand"),
                "is_self": original_info.get("is_self"),
                "original_address_history": original_info.get("address"),

                # 원본 전체 metadata
                "original_metadata": original_info,
            })
            continue

        for item in address_history:
            if not isinstance(item, list) or len(item) < 2:
                failed_records.append({
                    "station_id": station_id,
                    "start_date": None,
                    "address": None,
                    "message": f"address 항목 형식 오류: {item}",
                    "selected_source": None,

                    "station_name": original_info.get("station_name"),
                    "region": original_info.get("region"),
                    "brand": original_info.get("brand"),
                    "is_self": original_info.get("is_self"),
                    "original_address_history": original_info.get("address"),
                    "raw_address_item": item,

                    "original_metadata": original_info,
                })
                continue

            start_date = str(item[0]).strip()
            addr = normalize_space(item[1])

            if not addr:
                location_history.append([start_date, None, None])
                failed_records.append({
                    "station_id": station_id,
                    "start_date": start_date,
                    "address": addr,
                    "message": "주소값이 비어 있음",
                    "selected_source": None,

                    "station_name": original_info.get("station_name"),
                    "region": original_info.get("region"),
                    "brand": original_info.get("brand"),
                    "is_self": original_info.get("is_self"),
                    "original_address_history": original_info.get("address"),
                    "raw_address_item": item,

                    "original_metadata": original_info,
                })
                continue

            final_result = final_cache.get(addr)
            if final_result is None or not final_result.get("ok"):
                location_history.append([start_date, None, None])
                failed_records.append({
                    "station_id": station_id,
                    "start_date": start_date,
                    "address": addr,
                    "message": None if final_result is None else final_result.get("message"),
                    "selected_source": None if final_result is None else final_result.get("selected_source"),

                    "station_name": original_info.get("station_name"),
                    "region": original_info.get("region"),
                    "brand": original_info.get("brand"),
                    "is_self": original_info.get("is_self"),
                    "original_address_history": original_info.get("address"),
                    "roadAddr_if_any": None if final_result is None else final_result.get("standardized_roadAddr"),
                    "raw_final_result": final_result,

                    "original_metadata": original_info,
                })
                continue

            location_history.append([
                start_date,
                round(final_result["latitude"], 8) if final_result["latitude"] is not None else None,
                round(final_result["longitude"], 8) if final_result["longitude"] is not None else None,
            ])

        info["location"] = location_history

    if SAVE_OUTPUT_JSON:
        save_json(output_metadata, OUTPUT_PATH)
        log(f"[저장 완료] {OUTPUT_PATH}")

    final_success_cnt = 0
    final_total_cnt = 0
    for info in output_metadata.values():
        for item in info.get("location", []):
            final_total_cnt += 1
            if isinstance(item, list) and len(item) >= 3 and item[1] is not None and item[2] is not None:
                final_success_cnt += 1

    log("완료")
    log(f"  - 최종 성공 location 수: {final_success_cnt:,}/{final_total_cnt:,}")
    log(f"  - 실패 레코드 수: {len(failed_records):,}")
    log(f"  - 충돌/검토 레코드 수: {len(conflict_records):,}")

### 2-7. 전국 좌표 데이터 점검

In [ ]:
# =========================
# 0. 환경설정
# =========================
import os
import json
import sys
import subprocess
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# geopandas가 없으면 설치 시도
try:
    import geopandas as gpd
except Exception:
    print("[설치] geopandas/pyogrio 설치 시작", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "geopandas", "pyogrio", "shapely", "fiona"])
    import geopandas as gpd

# =========================
# 1. 함수 정의
# =========================
def log(msg):
    print(msg, flush=True)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def to_float_or_none(x):
    try:
        if x is None:
            return None
        return float(x)
    except Exception:
        return None

def nfc(text):
    if text is None:
        return text
    return unicodedata.normalize("NFC", str(text))

def collect_points_from_region_json(region_name, json_path):
    """
    metadata__latlon.json 에서
    None이 아닌 location만 뽑아 점 데이터로 반환
    """
    data = load_json(json_path)

    records = []
    summary = {
        "region": nfc(region_name),
        "station_count": len(data),
        "location_total": 0,
        "valid_point_count": 0,
        "none_count": 0,
        "empty_location_station_count": 0,
    }

    for station_id, info in data.items():
        location_history = info.get("location", [])

        if not isinstance(location_history, list) or len(location_history) == 0:
            summary["empty_location_station_count"] += 1
            continue

        station_name = info.get("station_name", [])
        address_history = info.get("address", [])

        for item in location_history:
            summary["location_total"] += 1

            if not isinstance(item, list) or len(item) < 3:
                summary["none_count"] += 1
                continue

            start_date = item[0]
            lat = to_float_or_none(item[1])   # 저장 순서: [시작일, 위도, 경도]
            lon = to_float_or_none(item[2])

            if lat is None or lon is None:
                summary["none_count"] += 1
                continue

            # 한국 범위 대략 검증
            if not (33.0 <= lat <= 39.5 and 124.0 <= lon <= 132.5):
                summary["none_count"] += 1
                continue

            records.append({
                "region": nfc(region_name),
                "station_id": station_id,
                "start_date": start_date,
                "latitude": lat,
                "longitude": lon,
                "station_name": station_name,
                "address_history": address_history,
            })
            summary["valid_point_count"] += 1

    return records, summary

def load_all_region_points(base_dir):
    """
    모든 지역 폴더의 metadata__latlon.json 읽기
    """
    all_records = []
    summaries = []
    found_files = []

    region_dirs = sorted([p for p in Path(base_dir).iterdir() if p.is_dir()])

    log(f"[스캔] 지역 폴더 수: {len(region_dirs)}")

    for region_dir in region_dirs:
        region_name = nfc(region_dir.name)
        json_path = region_dir / "metadata__latlon.json"

        if not json_path.exists():
            log(f"[건너뜀] {region_name}: metadata__latlon.json 없음")
            continue

        found_files.append(str(json_path))
        log(f"[로드] {region_name}: {json_path}")

        records, summary = collect_points_from_region_json(region_name, json_path)
        all_records.extend(records)
        summaries.append(summary)

        log(
            f"  - station={summary['station_count']:,}, "
            f"location_total={summary['location_total']:,}, "
            f"valid={summary['valid_point_count']:,}, "
            f"none_or_invalid={summary['none_count']:,}, "
            f"empty_location_station={summary['empty_location_station_count']:,}"
        )

    points_df = pd.DataFrame(all_records)
    summary_df = pd.DataFrame(summaries)

    return points_df, summary_df, found_files

def load_korea_background():
    """
    가능하면 배경 지도 불러오기
    실패하면 None 반환
    """
    try:
        # geopandas 기본 내장 world dataset
        world_path = gpd.datasets.get_path("naturalearth_lowres")
        world = gpd.read_file(world_path)

        # 한반도 주변만 crop
        world_crop = world.cx[123:133, 32:40.5].copy()

        # 한국/북한 후보명들
        korea_names = ["South Korea", "Republic of Korea", "Korea, Rep."]
        north_korea_names = ["North Korea", "Korea, Dem. Rep."]

        south_korea = world_crop[world_crop["name"].isin(korea_names)].copy()
        north_korea = world_crop[world_crop["name"].isin(north_korea_names)].copy()

        return world_crop, south_korea, north_korea

    except Exception as e:
        log(f"[배경지도 불러오기 실패] {repr(e)}")
        return None, None, None

def plot_points_on_korea(points_df, summary_df):
    if points_df.empty:
        raise ValueError("표시할 유효 좌표가 없습니다.")

    world_crop, south_korea, north_korea = load_korea_background()

    fig, ax = plt.subplots(figsize=(10, 12))

    # 배경 지도
    if world_crop is not None and len(world_crop) > 0:
        world_crop.plot(ax=ax, color="whitesmoke", edgecolor="gray", linewidth=0.6, zorder=1)

        if north_korea is not None and len(north_korea) > 0:
            north_korea.plot(ax=ax, color="gainsboro", edgecolor="gray", linewidth=0.7, zorder=2)

        if south_korea is not None and len(south_korea) > 0:
            south_korea.plot(ax=ax, color="white", edgecolor="black", linewidth=0.9, zorder=3)
    else:
        log("[안내] 배경지도 없이 점만 표시합니다.")

    # 지역별 색상
    regions = sorted(points_df["region"].dropna().unique().tolist())
    cmap = plt.cm.get_cmap("tab20", len(regions))
    color_map = {region: cmap(i) for i, region in enumerate(regions)}

    # 지역별 점 찍기
    for region in regions:
        temp = points_df[points_df["region"] == region]
        ax.scatter(
            temp["longitude"],
            temp["latitude"],
            s=8,
            alpha=0.8,
            label=f"{region} ({len(temp):,})",
            color=color_map[region],
            zorder=10
        )

    # 한반도 범위
    ax.set_xlim(124, 132.3)
    ax.set_ylim(33, 39.7)

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Gas Station Points by Region (Non-None Lat/Lon Only)")
    ax.grid(True, linestyle="--", alpha=0.3)

    # 범례
    ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        fontsize=8,
        frameon=True,
        ncol=1
    )

    plt.tight_layout()
    plt.show()

    return fig, ax

# =========================
# 2. 사용자정의 & 실행
# =========================
BASE_DIR = os.path.join(
    PROCESSED_PATH,
    "additional_data",
    "gas_station_prices_by_region"
)

log("[1/5] 전체 지역 metadata__latlon.json 스캔 시작")
points_df, summary_df, found_files = load_all_region_points(BASE_DIR)

log("[2/5] 파일 수 요약")
log(f"  - 읽은 파일 수: {len(found_files):,}")

log("[3/5] 점 데이터 요약")
log(f"  - 전체 유효 점 수: {len(points_df):,}")
if not summary_df.empty:
    print(summary_df.sort_values("region").to_string(index=False), flush=True)

log("[4/5] 지역별 점 개수")
region_counts = (
    points_df.groupby("region")
    .size()
    .reset_index(name="point_count")
    .sort_values(["point_count", "region"], ascending=[False, True])
)
print(region_counts.to_string(index=False), flush=True)

log("[5/5] 플롯 시작")
fig, ax = plot_points_on_korea(points_df, summary_df)

## 3. 개별공시지가

원본 프로젝트에서는 브이월드 계열 개별공시지가 데이터를 로컬 외부 프로그램으로 처리했습니다. 이 노트북에는 해당 로컬 처리 코드를 포함하지 않으며, 격자 단계에서는 처리 완료된 개별공시지가 결과 파일을 입력으로 사용합니다.